# Augmented clean12 ECG images on EC2, all in Account A: one paper/photo sheet per record

The single-account version of `Clean12_Augmented_ECG_Images_EC2.ipynb`. It reads the **clean12 signal dataset**, `ECG_500Hz_clean12.tar.gz`: 113,358 native-500 Hz WFDB records in Account A's S3. It writes **one augmented ECG sheet (1024x768 PNG) per record**, so the dataset ends with 113,358 images for 113,358 records. Each image path mirrors its record path (`images/<site>/WFDB/.../<record>.png`), the same layout as the clean12 sheets, so clean and augmented sheets match on `image_path`.

The output tar and its `.sha256` are uploaded to `s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12_augmented/` in the same bucket (`S3_UPLOAD_PREFIX`). After the upload is verified, **the instance terminates itself**. That happens only if every pipeline cell before it finished and no cell in this kernel raised an error (§11).

The images are identical to the two-account notebook's: same kit settings, augmentation, balanced plan and seed (all copied from it when this notebook is built).

**Per record:**
1. The record is rendered with the **ECG-image-kit**, using the same settings and the same code as `Clean12_ECG_Signal_Images_EC2.ipynb` (both copied from that notebook when this one is built). This gives the 2200x1700 grey sheet the clean12 images were made from.
2. That sheet goes through the image pipeline's **stage-4 paper/photo augmentation**: `stage4_augment.py`, the code behind the `augmented_<grid>_tilt<deg>.png` previews in `correct_prediction_samples`. Its wrinkled-paper texture and fold creases are **stronger** than stage 4 (table below).
3. JPEG q85 round trip, then the clean view's own 1024x768 fit.

The copy of stage 4 in `augment_worker.py` reproduces `stage4_augment.augment` pixel for pixel when run with the stage-4 settings. Only the wrinkle and crease strengths were raised:

| Setting | Stage 4 | This notebook |
|---|---|---|
| wrinkle texture strength | 0.30–0.62 | 0.65–1.00 |
| wrinkle shading limits | ×0.86–1.10 | ×0.76–1.14 |
| crease amplitude | 0.06–0.13 | 0.16–0.26 |
| crease shading limits | ×0.88–1.14 | ×0.76–1.22 |
| crease width (fraction of the sheet's long side) | 0.006–0.020 | 0.004–0.012 (sharper) |
| paper bowing | 0.02–0.06 | 0.03–0.08 |

**Equal representation.** These factors are assigned by plan, not by chance. Grid red is skipped because it is the clean view's grid.

| Factor | Levels | Share each |
|---|---|---|
| grid colour | orange, pink, grey | 1/3 |
| ink strength | 0.72–0.82 (faded), 0.82–0.92, 0.92–1.02 (fresh) | 1/3 |
| tilt | six 1° bins across −3°…+3° | 1/6 |
| perspective | off / on | 1/2 |
| fold creases | off / on | 1/2 |
| edge shadow | off / on | 1/2 |
| blur | off / on | 1/2 |

Every sheet gets wrinkled-paper texture (at the stronger setting above), uneven lighting, colour tint, brightness, contrast, paper-level normalisation and sensor noise, each at a random strength within its range.

How the plan is assigned:
- Levels are dealt round-robin inside each `(site, label set)` stratum, ignoring the labels themselves.
- Each factor uses its own seeded shuffle, so factors are assigned independently of each other.
- Result: level counts are within ±1 overall and within every label combination. §7 writes the audit and stops if it fails.

## Setup: data, EC2 and output in Account A

Account A owns `cardiosentry-fcmae-758821198776-us-east-1-an` (the account id is part of the bucket's name), and the instance runs in Account A too. There is no cross-account grant and no bucket-policy change.

**Files to copy to the instance**, all three from this notebook's folder in the repo, into one folder:
- this notebook;
- `ecg_image_v1_code.tar.gz`;
- `ecg_image_v1_code.tar.gz.sha256`.

`ecg_image_v1_code.tar.gz` is the ECG-image-kit itself: the program that draws the ECG from the signals, plus the wrinkled-paper photographs. If you would rather not copy it, put both kit files next to the signals (`S3_KIT_SOURCE`); §4 downloads them if no local copy is found.

**Instance role.** Create the EC2 role (with its instance profile) `cardiosentry-augment-ec2` with this policy, then launch the instance with it:
```json
{
  "Version": "2012-10-17",
  "Statement": [
    {"Sid": "ReadClean12", "Effect": "Allow", "Action": "s3:GetObject",
     "Resource": "arn:aws:s3:::cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/*"},
    {"Sid": "WriteAugmented", "Effect": "Allow",
     "Action": ["s3:PutObject", "s3:GetObject", "s3:DeleteObject", "s3:AbortMultipartUpload"],
     "Resource": "arn:aws:s3:::cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12_augmented/*"},
    {"Sid": "TerminateItself", "Effect": "Allow", "Action": "ec2:TerminateInstances",
     "Resource": "arn:aws:ec2:us-east-1:758821198776:instance/*",
     "Condition": {"ArnEquals": {"ec2:InstanceProfile": "arn:aws:iam::758821198776:instance-profile/cardiosentry-augment-ec2"}}}
  ]
}
```
Notes:
- Within one account the role policy is enough; the bucket policy needs no change unless it has an explicit `Deny`.
- If the bucket's default encryption is **SSE-KMS with a customer-managed key**, also allow `kms:Decrypt` and `kms:GenerateDataKey` on that key. The default SSE-S3 encryption needs nothing more.
- The condition limits termination to instances launched with this profile.

**Instance.**
- Type: **c8a.4xlarge** in **us-east-1**, the same region as the data.
- Storage: a **250 GB gp3** volume at 1,000 MB/s.
- Run Jupyter inside `tmux` (or with `nohup`), from the folder holding the three files.
- Run All.

Every stage can be restarted. A rerun reuses:
- the verified archive;
- the extracted tree;
- every sheet already rendered, as recorded in `WORK_DIR/generation_progress.csv`.

In [ ]:
# 1. Configuration: edit before running any later cell.
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

from pathlib import Path

# Account A (owns the bucket and runs this instance): the clean12 signal dataset. Same object as
# https://cardiosentry-fcmae-758821198776-us-east-1-an.s3.us-east-1.amazonaws.com/CardioSentry/dataset/clean12/ECG_500Hz_clean12.tar.gz
S3_SIGNAL_PREFIX = "s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/"
S3_SOURCE = S3_SIGNAL_PREFIX + "ECG_500Hz_clean12.tar.gz"
S3_SHA256_SOURCE = S3_SIGNAL_PREFIX + "ECG_500Hz_clean12.tar.gz.sha256"
# Fallback only: the kit bundle normally sits next to this notebook on the instance.
S3_KIT_SOURCE = S3_SIGNAL_PREFIX + "ecg_image_v1_code.tar.gz"
S3_KIT_SHA256_SOURCE = S3_SIGNAL_PREFIX + "ecg_image_v1_code.tar.gz.sha256"

# Account A, same bucket: where the augmented archive is uploaded.
S3_UPLOAD_PREFIX = "s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12_augmented/"
ARCHIVE_FORMAT = "tar"           # PNG is already compressed
ARCHIVE_BASENAME = "clean_ecg_signal_images_clean12_augmented"   # -> <prefix><basename>.tar

WORK_DIR = Path.home() / "ecg_clean12_augment"
LOCAL_ARCHIVE = WORK_DIR / "input" / "ECG_500Hz_clean12.tar.gz"
INPUT_DIR = WORK_DIR / "input" / "ECG_500Hz_clean12"
OUTPUT_DIR = WORK_DIR / "clean_ecg_signal_images_clean12_augmented"

EXPECTED_RECORDS = 113_358
CONDITIONS = ["SB", "STACH", "LBBB", "LQT", "AF", "AFL",
              "3AVB", "2AVB", "1AVB", "STEMI", "STE", "NORM"]
# Kit/render settings: copied from Clean12_ECG_Signal_Images_EC2.ipynb at build
# time, so every grey master is the master its clean12 sheet was made from.
EXPECTED_FS = 500
EXPECTED_SAMPLES = 5000
MAX_DURATION_S = 10.0
KIT_SEED = 20260918
KIT_FIXED_FONT = 'Arial.ttf'
KIT_RENDER_FLAGS = ['--num_columns', '4', '--full_mode', 'II', '-r', '200', '--pad_inches', '0', '--calibration_pulse', '1', '--random_bw', '1']
KIT_ENV = {'ECG_ROW_GAP': '4.4', 'ECG_RHYTHM_GAP': '5.5', 'ECG_BOTTOM_RESERVE': '1.9'}
PNG_COMPRESS_LEVEL = 3
AUGMENT_SEED = 20261023          # plan shuffles + per-record effect draws
EST_IMAGE_MIB = 1.5              # measured 1.13-1.49 MiB per augmented sheet

# One worker per vCPU (c8a: 1 vCPU = 1 core). A worker runs one kit process for
# a whole batch (amortises the kit's 1-3 s import), then augments its masters.
WORKERS = max(1, os.cpu_count() or 1)
BATCH_SIZE = 48
MAX_PENDING_BATCHES = 2 * WORKERS
PROGRESS_EVERY = 1000
RETRY_PASSES = 1                 # failed records are retried once before giving up
KIT_SCRATCH_DIR = (Path("/dev/shm") / "ecg_clean12_augment_scratch"
                   if os.access("/dev/shm", os.W_OK) else WORK_DIR / "kit_scratch")
S3_TRANSFER = dict(max_concurrency=32, multipart_chunksize=64 * 2**20, use_threads=True)

# Terminate this instance once the archive is uploaded and verified (§11), and
# only if every pipeline cell completed and no cell raised in this kernel.
AUTO_TERMINATE = True
TERMINATE_GRACE_SECONDS = 60     # interrupt the kernel within this window to cancel

# Safety record for §11. Re-running THIS cell starts a fresh run: both reset.
STAGES_DONE = set()              # each pipeline cell adds its name when it completes
CELL_ERRORS = []                 # every cell that raised in this kernel since this cell
try:
    _ipython = get_ipython()
except NameError:                # plain Python (no IPython): only STAGES_DONE applies
    _ipython = None
if _ipython is not None:
    for _callback in list(_ipython.events.callbacks["post_run_cell"]):
        if getattr(_callback, "__name__", "") == "_record_cell_error":
            _ipython.events.unregister("post_run_cell", _callback)

    def _record_cell_error(result):
        if not result.success:
            error = result.error_in_exec or result.error_before_exec
            CELL_ERRORS.append(f"{type(error).__name__}: {error}"[:300])

    _ipython.events.register("post_run_cell", _record_cell_error)

assert WORKERS >= 1 and BATCH_SIZE >= 1 and MAX_PENDING_BATCHES >= WORKERS
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR, "| workers:", WORKERS, "| batch:", BATCH_SIZE)
print("Source:", S3_SOURCE)
print("Upload:", S3_UPLOAD_PREFIX + ARCHIVE_BASENAME + "." + ARCHIVE_FORMAT, "| auto-terminate:", AUTO_TERMINATE)

## 2. Install and verify runtime dependencies

This is the clean12 render notebook's install cell, unchanged: the kit's batch driver runs on this kernel's Python, and it requires `wfdb>=4.3`. `pandas` is also used by the plan audit.

In [ ]:
# Install ALL runtime packages into this kernel's environment BEFORE any
# third-party import. Unconditional (a fast no-op when already satisfied) so a
# half-present environment can never slip through a presence check and fail
# later at `import scipy` with ModuleNotFoundError.
%pip install --quiet boto3 numpy scipy Pillow matplotlib pandas qrcode pyyaml wfdb
import importlib as _package_importlib
import subprocess as _package_subprocess
import sys as _kernel_sys

# boto3 = S3 I/O; numpy/scipy/Pillow = worker staging + recolour + resize;
# matplotlib/pandas/qrcode/pyyaml/wfdb = the kit batch-driver imports.
_module_to_spec = {"boto3": "boto3", "numpy": "numpy", "scipy": "scipy",
                   "PIL": "Pillow", "matplotlib": "matplotlib",
                   "pandas": "pandas", "qrcode": "qrcode",
                   "yaml": "pyyaml", "wfdb": "wfdb>=4.3"}

# wfdb>=4.3 is REQUIRED (4.1.x dies at import on pandas 3): upgrade a stale
# copy BEFORE it is ever imported in this process (upgrading after a failed
# import mixes versions and needs a kernel restart instead).
try:
    from importlib.metadata import version as _dist_version
    _wfdb_have = _dist_version("wfdb")
    _wfdb_old = tuple(int(p) for p in _wfdb_have.split(".")[:2]) < (4, 3)
except Exception:
    _wfdb_have, _wfdb_old = "?", False
if _wfdb_old:
    if "wfdb" in _kernel_sys.modules:
        raise RuntimeError(
            f"wfdb {_wfdb_have} is already imported and too old (need >=4.3): "
            "restart the kernel and re-run from §1.")
    print(f"Upgrading stale wfdb {_wfdb_have} to wfdb>=4.3 ...", flush=True)
    _package_subprocess.check_call(
        [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
         "--retries", "5", "--timeout", "300", "wfdb>=4.3"])
    _package_importlib.invalidate_caches()

# Verify every import; an unimportable module gets one automatic repair
# install (covers detection gaps and broken copies) instead of failing later.
for _mod, _spec in _module_to_spec.items():
    try:
        _package_importlib.import_module(_mod)
    except Exception as exc:
        print(f"{_mod} not importable ({type(exc).__name__}); "
              f"repair-installing {_spec} ...", flush=True)
        _package_subprocess.check_call(
            [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
             "--retries", "5", "--timeout", "300", _spec])
        _package_importlib.invalidate_caches()
        _package_importlib.import_module(_mod)
print("Runtime packages verified:",
      ", ".join(sorted(_module_to_spec)), flush=True)

import csv
from collections import Counter
import hashlib
import importlib
import json
import math
import multiprocessing as mp
import shutil
import sys
import tarfile
import time
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3
import numpy as np
import scipy
from scipy.io import loadmat as _checked_loadmat
from PIL import Image
from boto3.s3.transfer import TransferConfig

print("Python", sys.version.split()[0], "| boto3", boto3.__version__,
      "| NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| Pillow", Image.__version__)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import subprocess
import urllib.request
from collections import defaultdict
from pathlib import PurePosixPath

import pandas as pd
from botocore.exceptions import ClientError
STAGES_DONE.add("install")

## 3. Download, verify and extract the signals

This is the clean12 render notebook's download cell, unchanged:
- it reads the `.sha256` sidecar, downloads the archive and hashes all of it;
- it extracts safely, rejecting links and any path outside the destination;
- a completion marker lets a rerun skip the work.

The first S3 call is where a missing permission shows up. `AccessDenied` here means the instance role lacks the `ReadClean12` statement (top cell), or the bucket's KMS key does not let the role decrypt.

In [ ]:
def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or ":" in parsed.netloc or not parsed.path.strip("/"):
        raise ValueError(f"Expected s3://bucket/key, got {uri!r}")
    return parsed.netloc, parsed.path.lstrip("/")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            block = stream.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

from botocore.exceptions import NoCredentialsError, PartialCredentialsError

aws_session = boto3.Session()
try:
    aws_credentials = aws_session.get_credentials()
except (NoCredentialsError, PartialCredentialsError) as exc:
    aws_credentials = None
if aws_credentials is None:
    raise RuntimeError(
        "No AWS credentials are available to this Jupyter kernel. Attach an EC2 IAM "
        "role to this instance: EC2 console > Instances > select instance > Actions > "
        "Security > Modify IAM role. Grant read access to the dataset archive and "
        "checksum, plus write/read access under the S3_UPLOAD_PREFIX. Then rerun this cell."
    )
print("AWS credential source:", aws_credentials.method)
s3 = aws_session.client("s3")
src_bucket, src_key = split_s3_uri(S3_SOURCE)
sha_bucket, sha_key = split_s3_uri(S3_SHA256_SOURCE)
try:
    checksum_object = s3.get_object(Bucket=sha_bucket, Key=sha_key)
except (NoCredentialsError, PartialCredentialsError) as exc:
    raise RuntimeError(
        "AWS credentials could not be used. Check the EC2 instance IAM role and "
        "instance metadata access, then rerun this cell."
    ) from exc
if int(checksum_object["ContentLength"]) > 1024:
    raise ValueError("S3 SHA-256 sidecar is unexpectedly large")
checksum_fields = checksum_object["Body"].read().decode("utf-8").strip().split()
if len(checksum_fields) != 2 or checksum_fields[1].lstrip("*") != LOCAL_ARCHIVE.name:
    raise ValueError("SHA-256 sidecar must name the dataset archive")
expected_sha256 = checksum_fields[0].lower()
if len(expected_sha256) != 64 or any(char not in "0123456789abcdef" for char in expected_sha256):
    raise ValueError("SHA-256 sidecar does not contain a valid digest")

src_info = s3.head_object(Bucket=src_bucket, Key=src_key)
source_bytes = int(src_info["ContentLength"])
source_identity = {"uri": S3_SOURCE, "sha256_uri": S3_SHA256_SOURCE,
                   "sha256": expected_sha256, "bytes": source_bytes,
                   "etag": src_info.get("ETag", "")}
source_sidecar = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".source.json")
archive_matches = (LOCAL_ARCHIVE.is_file() and LOCAL_ARCHIVE.stat().st_size == source_bytes
                   and file_sha256(LOCAL_ARCHIVE) == expected_sha256)
if archive_matches:
    print("Reusing SHA-256 verified source archive:", LOCAL_ARCHIVE)
else:
    partial = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".part")
    partial.unlink(missing_ok=True)
    print(f"Downloading {S3_SOURCE} ({source_bytes / 2**30:.2f} GiB) ...")
    s3.download_file(src_bucket, src_key, str(partial),
                     Config=TransferConfig(**S3_TRANSFER))
    if partial.stat().st_size != source_bytes:
        raise IOError("Downloaded byte count differs from S3 ContentLength")
    if file_sha256(partial) != expected_sha256:
        partial.unlink(missing_ok=True)
        raise IOError("Downloaded archive SHA-256 does not match S3 sidecar")
    partial.replace(LOCAL_ARCHIVE)
    print("Downloaded and SHA-256 verified:", LOCAL_ARCHIVE)
source_sidecar.write_text(json.dumps(source_identity, indent=2) + "\n")

extraction_root = INPUT_DIR.parent.resolve()
marker = extraction_root / ".ecg_clean12_extraction_complete.json"
marker_value = {"source_uri": S3_SOURCE, "archive_sha256": expected_sha256,
                "archive_bytes": source_bytes,
                "archive_mtime": LOCAL_ARCHIVE.stat().st_mtime_ns}
already_extracted = False
if marker.is_file():
    try:
        already_extracted = json.loads(marker.read_text()) == marker_value and INPUT_DIR.is_dir()
    except (OSError, ValueError):
        pass
if already_extracted:
    print("Reusing extracted clean12 ECG tree:", INPUT_DIR)
else:
    marker.unlink(missing_ok=True)
    print("Extracting to", extraction_root, "...")
    member_count = 0
    from pathlib import PurePosixPath
    with tarfile.open(LOCAL_ARCHIVE, "r:gz") as archive:
        for member in archive:
            rel = PurePosixPath(member.name)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts or rel.parts[0] != INPUT_DIR.name:
                raise ValueError(f"Unsafe/unexpected archive path: {member.name!r}")
            dest = extraction_root.joinpath(*rel.parts)
            if not dest.resolve(strict=False).is_relative_to(extraction_root):
                raise ValueError(f"Archive path resolves outside destination: {member.name!r}")
            if member.isdir():
                dest.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                if dest.is_symlink():
                    raise ValueError(f"Refusing to overwrite symlink: {dest}")
                dest.parent.mkdir(parents=True, exist_ok=True)
                stream = archive.extractfile(member)
                if stream is None:
                    raise IOError(f"Cannot read archive member {member.name}")
                with stream, dest.open("wb") as out:
                    shutil.copyfileobj(stream, out, length=1024 * 1024)
                if dest.stat().st_size != member.size:
                    raise IOError(f"Incomplete extracted member {member.name}")
                member_count += 1
            else:
                raise ValueError(f"Links/special files are disallowed: {member.name!r}")
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f"Archive did not create {INPUT_DIR}")
    marker.write_text(json.dumps(marker_value, indent=2) + "\n")
    print(f"Extracted {member_count:,} files")

def ec2_identity():
    """This instance's id, region and type from IMDSv2 (used by §8 and §11)."""
    token = urllib.request.urlopen(urllib.request.Request(
        "http://169.254.169.254/latest/api/token", method="PUT",
        headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"}), timeout=3).read().decode()
    document = urllib.request.urlopen(urllib.request.Request(
        "http://169.254.169.254/latest/dynamic/instance-identity/document",
        headers={"X-aws-ec2-metadata-token": token}), timeout=3).read().decode()
    doc = json.loads(document)
    return {"instance_id": doc["instanceId"], "region": doc["region"], "instance_type": doc["instanceType"]}

STAGES_DONE.add("download")

## 4. Kit bundle: verify, extract, patch gate

This is the clean12 render notebook's kit cell, unchanged. It finds `ecg_image_v1_code.tar.gz` next to this notebook, or falls back to `S3_KIT_SOURCE`. The bundle must match its `.sha256` sidecar, and rendering is refused unless every kit patch marker and the fixed font are present. The wrinkled-paper photographs come from the same bundle.

In [ ]:
KIT_BUNDLE = None
_kit_search = [Path.cwd() / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "ecg_image_v1_code.tar.gz"]
for _ancestor in [Path.cwd(), *list(Path.cwd().parents)[:4]]:
    _kit_search.append(_ancestor / "dataset_generation_code" / "ecg_image_v1_code.tar.gz")
for _cand in _kit_search:
    if _cand.is_file():
        KIT_BUNDLE = _cand
        break
if KIT_BUNDLE is None:
    # No local copy — pull the kit bundle from S3 with the same sidecar-pinned
    # discipline as the source archive, then fall through to the checks below.
    try:
        print("No local kit bundle; downloading from", S3_KIT_SOURCE, "...", flush=True)
        _kit_dest = WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz"
        _kit_dest.parent.mkdir(parents=True, exist_ok=True)
        _kb, _kk = split_s3_uri(S3_KIT_SOURCE)
        _sb, _sk = split_s3_uri(S3_KIT_SHA256_SOURCE)
        _sobj = s3.get_object(Bucket=_sb, Key=_sk)
        if int(_sobj["ContentLength"]) > 1024:
            raise ValueError("Kit SHA-256 sidecar is unexpectedly large")
        _raw = _sobj["Body"].read()
        (_kit_dest.parent / (_kit_dest.name + ".sha256")).write_bytes(_raw)
        _fields = _raw.decode("utf-8").strip().split()
        if len(_fields) != 2 or _fields[1].lstrip("*") != _kit_dest.name:
            raise ValueError("Kit SHA-256 sidecar must name the kit bundle")
        _want = _fields[0].lower()
        if len(_want) != 64 or any(c not in "0123456789abcdef" for c in _want):
            raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
        _kinfo = s3.head_object(Bucket=_kb, Key=_kk)
        _kbytes = int(_kinfo["ContentLength"])
        if not (_kit_dest.is_file() and _kit_dest.stat().st_size == _kbytes
                and file_sha256(_kit_dest) == _want):
            _kpart = _kit_dest.with_name(_kit_dest.name + ".part")
            _kpart.unlink(missing_ok=True)
            print(f"Downloading {S3_KIT_SOURCE} ({_kbytes / 2**20:.1f} MiB) ...", flush=True)
            s3.download_file(_kb, _kk, str(_kpart),
                             Config=TransferConfig(**S3_TRANSFER))
            if _kpart.stat().st_size != _kbytes:
                raise IOError("Downloaded kit byte count differs from S3 ContentLength")
            if file_sha256(_kpart) != _want:
                _kpart.unlink(missing_ok=True)
                raise IOError("Downloaded kit SHA-256 does not match S3 sidecar")
            _kpart.replace(_kit_dest)
            print("Downloaded and SHA-256 verified:", _kit_dest, flush=True)
        else:
            print("Reusing SHA-256 verified kit bundle:", _kit_dest, flush=True)
        KIT_BUNDLE = _kit_dest
    except Exception as exc:
        raise FileNotFoundError(
            "ecg_image_v1_code.tar.gz not found locally and the S3 fallback failed "
            f"({type(exc).__name__}: {exc}). Searched: " +
            ", ".join(str(p) for p in _kit_search) +
            f"; S3: {S3_KIT_SOURCE}") from exc
_kit_sidecar = Path(str(KIT_BUNDLE) + ".sha256")
if not _kit_sidecar.is_file():
    raise FileNotFoundError(f"Kit SHA-256 sidecar missing: {_kit_sidecar}")
_kit_want = _kit_sidecar.read_text().split()[0].lower()
if len(_kit_want) != 64 or any(c not in "0123456789abcdef" for c in _kit_want):
    raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
KIT_BUNDLE_SHA256 = file_sha256(KIT_BUNDLE)
if KIT_BUNDLE_SHA256 != _kit_want:
    raise IOError("Kit bundle SHA-256 does not match its sidecar — re-transfer the file")
print("Kit bundle verified:", KIT_BUNDLE)

KIT_ROOT = WORK_DIR / "kit"
_kit_marker = KIT_ROOT / ".kit_extraction_complete.json"
_kit_marker_value = {"bundle_sha256": KIT_BUNDLE_SHA256,
                     "bundle_bytes": KIT_BUNDLE.stat().st_size}
_kit_ready = False
if _kit_marker.is_file():
    try:
        _kit_ready = json.loads(_kit_marker.read_text()) == _kit_marker_value and KIT_ROOT.is_dir()
    except (OSError, ValueError):
        pass
if _kit_ready:
    print("Reusing extracted kit:", KIT_ROOT)
else:
    _kit_marker.unlink(missing_ok=True)
    print("Extracting kit to", KIT_ROOT, "...")
    shutil.rmtree(KIT_ROOT, ignore_errors=True)
    with tarfile.open(KIT_BUNDLE, "r:gz") as _archive:
        try:
            _archive.extractall(KIT_ROOT, filter="data")
        except TypeError:  # Python without PEP 706 filters; bundle is SHA-256 verified
            _archive.extractall(KIT_ROOT)
    _kit_marker.write_text(json.dumps(_kit_marker_value, indent=2) + "\n")
    print("Kit extracted")

KIT_DIR = str(KIT_ROOT / "ecg_image_v1_code" / "ecg-image-generator")
_KIT_SCRIPT = Path(KIT_DIR) / "gen_ecg_images_from_data_batch.py"
if not _KIT_SCRIPT.is_file():
    raise FileNotFoundError(f"Kit batch driver missing: {_KIT_SCRIPT}")
if not (Path(KIT_DIR) / "Fonts" / KIT_FIXED_FONT).is_file():
    raise FileNotFoundError(f"Fixed font {KIT_FIXED_FONT} missing from kit Fonts/")

# Patch gate (PLAN §6): refuse to render from unpatched kit code.
_kit_need = {
    "ecg_plot.py": ["PATCHED by image_pipeline/patch_kit.py",
                    "PATCHED-LEADNAME", "PATCHED-RHYTHMNAME",
                    "PATCHED-INKWIDTH", "PATCHED-LAYOUT", "PATCHED-PAPER"],
    "extract_leads.py": ["PATCHED-COLUMNS"],
    "gen_ecg_images_from_data_batch.py": ["PATCHED-NUMPY2"],
    "gen_ecg_image_from_data.py": ["PATCHED-LAZYDEPS", "PATCHED-FIXEDFONT"],
    "helper_functions.py": ["PATCHED-UNUSED-IMGAUG"],
}
_kit_missing = []
for _name, _markers in _kit_need.items():
    _text = (Path(KIT_DIR) / _name).read_text()
    for _mk in _markers:
        if _mk not in _text:
            _kit_missing.append(f"{_name}:{_mk}")
if _kit_missing:
    raise RuntimeError(f"Kit at {KIT_DIR} is not patched "
                       f"({len(_kit_missing)} markers missing, e.g. {_kit_missing[:3]}); refusing to render")
print(f"Kit patch gate OK ({KIT_DIR})")

TEXTURE_DIR = Path(KIT_DIR) / "CreasesWrinkles" / "wrinkles-dataset"
_textures = sorted(p.name for p in TEXTURE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
if len(_textures) != 19:
    raise RuntimeError(f"Expected the kit's 19 wrinkled-paper photographs, found {len(_textures)}")
print(f"{len(_textures)} wrinkled-paper photographs in {TEXTURE_DIR}")
STAGES_DONE.add("kit")

## 5. Discover and validate the 113,358 records

Records come from `manifest.csv`, never from globbing. Each header must be 12 leads × 5,000 samples at 500 Hz, and the label totals must equal `_checks/final_counts.csv`. These are the same checks the clean12 render notebook makes.

In [ ]:
with (INPUT_DIR / "manifest.csv").open(newline="", encoding="utf-8") as handle:
    source_rows = list(csv.DictReader(handle))
if len(source_rows) != EXPECTED_RECORDS or len({r["record_path"] for r in source_rows}) != EXPECTED_RECORDS:
    raise ValueError(f"manifest.csv lists {len(source_rows):,} rows / "
                     f"{len({r['record_path'] for r in source_rows}):,} unique; expected {EXPECTED_RECORDS:,}")

site_counts, label_totals, patients, records = Counter(), Counter(), set(), []
for row in source_rows:
    rel = PurePosixPath(row["record_path"])
    if (rel.is_absolute() or ".." in rel.parts or rel.suffix != ".hea"
            or rel.parts[0] != row["site"] or rel.parts[1] != "WFDB"):
        raise ValueError(f"Unexpected record_path: {row['record_path']!r}")
    header = INPUT_DIR.joinpath(*rel.parts)
    if not header.is_file():
        raise FileNotFoundError(f"Manifest record missing from extracted tree: {rel}")
    first = header.read_text(encoding="utf-8", errors="replace").split("\n", 1)[0].split()
    if (len(first) < 4 or first[1] != "12"
            or float(first[2].split("/")[0]) != EXPECTED_FS or int(first[3]) != EXPECTED_SAMPLES):
        raise ValueError(f"{rel}: header {' '.join(first[:4])!r} is not 12 x {EXPECTED_SAMPLES} @ {EXPECTED_FS} Hz")
    site_counts[row["site"]] += 1
    patients.add(row["patient_key"])
    for condition in CONDITIONS:
        label_totals[condition] += int(row[condition])
    records.append({"record_path": row["record_path"], "site": row["site"], "labels": row["labels"],
                    "image_path": (PurePosixPath("images") / rel.with_suffix(".png")).as_posix()})

with (INPUT_DIR / "_checks" / "final_counts.csv").open(newline="", encoding="utf-8") as handle:
    verified_totals = {row["condition"]: int(row["total"]) for row in csv.DictReader(handle)}
if verified_totals != dict(label_totals):
    raise ValueError(f"Label totals {dict(label_totals)} differ from _checks/final_counts.csv {verified_totals}")
record_by_path = {row["record_path"]: row for row in source_rows}
print(f"{len(records):,} records from {len(patients):,} patients | per site: {dict(sorted(site_counts.items()))}")
print("Records per condition:", dict(label_totals))
STAGES_DONE.add("records")

## 6. Worker modules

Two modules are written into `WORK_DIR`. The spawned pool processes import them, so `WORK_DIR` is put on `PYTHONPATH`.
- `ecg_clean_worker.py` is the clean12 render worker, verbatim: header parsing, millivolt loading, canonical `.dat` staging, the kit call, and the 1024x768 fit.
- `augment_worker.py` contains the stage-4 copy, the balanced planner, and the pool task. One task makes one kit call for a batch of records, then augments each grey sheet and writes it out.

In [ ]:
CLEAN_WORKER_SOURCE = r'''
"""Clean ECG worker: render with the real ECG-image-kit (PLAN §6).

Each batch is staged to canonical scratch WFDB .dat files (fmt 16, gain
1000, canonical lead order, int16 round-trip verified — this dodges the
kit's .mat gain bug, PLAN §0.5) and rendered by the kit batch driver
with the reference flags/env/fixed font. Grey masters are recoloured to
the canonical clean red grid (fixed LUT, PLAN §10) and resized LANCZOS
to 994x768 + 15 px paper-colour pads -> final 1024x768 PNG (PLAN §7.3).

Self-contained: stdlib + numpy + scipy + Pillow only. The kit itself
runs in a subprocess (kit_python, which carries the kit's own deps).
"""
import math
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from PIL import Image

LEADS = ("I", "II", "III", "aVR", "aVL", "aVF",
         "V1", "V2", "V3", "V4", "V5", "V6")
KIT_SCRIPT = "gen_ecg_images_from_data_batch.py"
KIT_TIMEOUT_S = 3600

# Clean-view recolour: fixed "red" palette (PLAN §10.3; port of
# v1_support.variants._lut with ink=(20,20,20), paper=red paper, sat=1.0).
GRID_MAJOR = (178.0, 86.0, 84.0)
GRID_MINOR = (228.0, 180.0, 176.0)
GRID_PAPER = (253.0, 249.0, 247.0)
GRID_INK = (20.0, 20.0, 20.0)
GRID_TRACE_MAX = 55
GRID_MAJOR_LEVEL = 102
GRID_MINOR_LEVEL = 191
GRID_PAPER_LEVEL = 250

FINAL_W, FINAL_H = 1024, 768
RESIZE_H = 768
MASTER_W, MASTER_H = 2200, 1700

CFG = None
INPUT = None
OUTPUT = None
_LUT = None


class InvalidRecordError(ValueError):
    pass


def init_worker(config, input_dir, output_dir):
    global CFG, INPUT, OUTPUT
    CFG = config
    INPUT = Path(input_dir)
    OUTPUT = Path(output_dir)
    for key in ("kit_dir", "kit_python", "kit_seed", "kit_fixed_font",
                "kit_render_flags", "kit_env", "kit_scratch_dir",
                "max_duration_s", "png_compress_level"):
        if key not in CFG:
            raise ValueError(f"render_config missing {key!r}")
    if not (Path(CFG["kit_dir"]) / KIT_SCRIPT).is_file():
        raise ValueError("kit batch driver missing — rerun the kit-bundle cell")


def _lut():
    major = np.asarray(GRID_MAJOR, dtype=np.float32)
    minor = np.asarray(GRID_MINOR, dtype=np.float32)
    paper = np.asarray(GRID_PAPER, dtype=np.float32)
    ink = np.asarray(GRID_INK, dtype=np.float32)

    def _sat(c):
        lum = float(np.dot(c, (0.299, 0.587, 0.114)))
        return np.clip(lum + (c - lum) * 1.0, 0, 255)

    major, minor = _sat(major), _sat(minor)
    xs = np.array([0, GRID_TRACE_MAX, GRID_MAJOR_LEVEL,
                   GRID_MINOR_LEVEL, GRID_PAPER_LEVEL, 255], dtype=np.float32)
    lut = np.empty((256, 3), dtype=np.float32)
    grid = np.arange(256, dtype=np.float32)
    for ch in range(3):
        ys = np.array([ink[ch] * 0.35, ink[ch], major[ch], minor[ch],
                       paper[ch], min(255.0, paper[ch] + 3)], dtype=np.float32)
        lut[:, ch] = np.interp(grid, xs, ys)
    return np.clip(lut, 0, 255)


def recolour_clean(master):
    """Fixed clean-red recolour of a grey kit master.

    The float LUT is truncated to uint8 once (what the kit-v1 per-pixel
    `_LUT[grey].astype(np.uint8)` did) and applied through Pillow point
    tables: bit-identical output, ~7x cheaper per image.
    """
    global _LUT
    if _LUT is None:
        lut = _lut().astype(np.uint8)
        _LUT = [lut[:, ch].tolist() for ch in range(3)]
    grey = master.convert("RGB").convert("L")
    return Image.merge("RGB", [grey.point(table) for table in _LUT])


def _paper_colour(arr):
    # 90th percentile of corner patches (not median): antialiased grid-halo
    # pixels drag a median toward tint; p90 recovers the paper white.
    h, w = arr.shape[:2]
    k = max(8, min(h, w) // 40)
    patches = np.concatenate([
        arr[:k, :k].reshape(-1, 3), arr[:k, -k:].reshape(-1, 3),
        arr[-k:, :k].reshape(-1, 3), arr[-k:, -k:].reshape(-1, 3)])
    return tuple(int(v) for v in np.percentile(patches, 90, axis=0))


def to_final(img):
    """LANCZOS fit to height 768 + symmetric paper-colour pad to 1024x768."""
    w, h = img.size
    scale = RESIZE_H / h
    new_w = round(w * scale)
    small = img.resize((new_w, RESIZE_H), Image.LANCZOS)
    if abs((w / h) - (new_w / RESIZE_H)) > 0.5 / RESIZE_H + 1e-9:
        raise InvalidRecordError(
            f"aspect drift: source {w}x{h} -> {new_w}x{RESIZE_H}")
    pad_total = FINAL_W - new_w
    if pad_total < 0:
        raise InvalidRecordError(
            f"resized width {new_w} exceeds final {FINAL_W}")
    left = pad_total // 2
    fill = _paper_colour(np.asarray(small.convert("RGB"), dtype=np.float32))
    canvas = Image.new("RGB", (FINAL_W, FINAL_H), fill)
    canvas.paste(small.convert("RGB"), (left, 0))
    if canvas.size != (FINAL_W, FINAL_H):
        raise InvalidRecordError(f"final size {canvas.size}")
    return canvas


def parse_header(path):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError as exc:
        raise InvalidRecordError(f"Unreadable header: {exc}") from exc
    if not lines:
        raise InvalidRecordError("Empty header")
    first = lines[0].split()
    if len(first) < 4 or "/" in first[0] or first[0] != path.stem:
        raise InvalidRecordError("Malformed/multi-segment record line")
    try:
        nleads = int(first[1])
        fs = float(first[2].split("/")[0])
        nsamp = int(first[3])
    except ValueError as exc:
        raise InvalidRecordError("Invalid record dimensions") from exc
    if nleads != 12 or not math.isfinite(fs) or fs <= 0 or nsamp <= 0:
        raise InvalidRecordError(
            f"Unsupported dimensions: {nleads} leads, fs={fs}, n={nsamp}")
    duration = nsamp / fs
    if duration > CFG["max_duration_s"] + 1e-6:
        raise InvalidRecordError(
            f"Duration {duration:.4f}s exceeds configured maximum")
    rows = []
    comments = {}
    for line in lines[1:]:
        if line.startswith("#"):
            body = line[1:].strip()
            if ":" in body:
                k, v = body.split(":", 1)
                comments[k.strip()] = v.strip()
            continue
        if not line.strip():
            continue
        f = line.split()
        if len(f) < 9:
            raise InvalidRecordError("Malformed signal line")
        name = f[0]
        if name != Path(name).name or name in (".", "..", "~"):
            raise InvalidRecordError("Unsafe signal filename")
        fmt = f[1]
        if fmt not in ("16", "16+24", "16x1+24"):
            raise InvalidRecordError(f"Unsupported signal format {fmt}")
        gain_field = f[2]
        if not gain_field.endswith("/mV"):
            raise InvalidRecordError(f"Expected /mV units, got {gain_field}")
        try:
            gain = float(gain_field.split("(")[0].split("/")[0])
            adc_zero = int(f[4])
        except ValueError as exc:
            raise InvalidRecordError("Invalid gain or ADC zero") from exc
        if not math.isfinite(gain) or gain <= 0:
            raise InvalidRecordError("Nonpositive/nonfinite ADC gain")
        if "(" in gain_field and gain_field.split("(", 1)[1].split(")", 1)[0] != str(adc_zero):
            raise InvalidRecordError("Header baseline/ADC zero disagree")
        rows.append({"file": name, "fmt": fmt, "gain": gain,
                     "zero": adc_zero, "lead": f[-1]})
    if len(rows) != 12:
        raise InvalidRecordError("Need exactly 12 signal lines")
    upper = {lead.upper(): lead for lead in LEADS}
    names = []
    for r in rows:
        key = r["lead"].strip().upper()
        if key not in upper:
            raise InvalidRecordError(f"Non-canonical lead {r['lead']!r}")
        names.append(upper[key])
    if sorted(names) != sorted(LEADS):
        raise InvalidRecordError("Lead set is not the canonical 12")
    if len({r["file"] for r in rows}) != 1 or len({r["fmt"] for r in rows}) != 1:
        raise InvalidRecordError("Multiple signal files/formats not supported")
    return rows, names, fs, nsamp, duration, comments


def load_mv(path, rows, names, nsamp):
    fmt = rows[0]["fmt"]
    sig_path = path.parent / rows[0]["file"]
    if not sig_path.is_file():
        raise InvalidRecordError(f"Missing signal file {rows[0]['file']}")
    if fmt == "16":
        n = 12 * nsamp
        raw = np.fromfile(str(sig_path), dtype="<i2", count=n).astype(np.int64)
        if raw.size != n:
            raise InvalidRecordError(
                f".dat holds {raw.size} samples, need {n}")
        raw = raw.reshape(nsamp, 12).T
    else:
        try:
            arr = np.asarray(loadmat(str(sig_path), variable_names=["val"])["val"])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            raise InvalidRecordError(f"Unreadable MAT val: {exc}") from exc
        if arr.shape == (nsamp, 12):
            arr = arr.T
        if arr.shape != (12, nsamp) or arr.dtype.kind not in "iu":
            raise InvalidRecordError(
                f"Invalid MAT val shape/type {arr.shape}/{arr.dtype}")
        raw = arr.astype(np.int64)
    if raw.shape != (12, nsamp):
        raise InvalidRecordError(f"Unexpected signal shape {raw.shape}")
    pos = {name: i for i, name in enumerate(names)}
    order = [pos[lead] for lead in LEADS]
    gains = np.array([rows[pos[lead]]["gain"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    zeros = np.array([rows[pos[lead]]["zero"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    # Necessary unit conversion only: digital ADC counts -> physical millivolts.
    mv = (raw[order, :].astype(np.float64) - zeros) / gains
    if not np.isfinite(mv).all():
        raise InvalidRecordError("Nonfinite physical signal")
    return mv


def stage_record(record_id, mv, fs, comments, staged_dir):
    """Write canonical scratch .dat + .hea for the kit; verify round-trip."""
    if mv.shape[0] != 12:
        raise InvalidRecordError(f"shape {mv.shape}, need (12, N)")
    n = mv.shape[1]
    raw = np.round(mv * 1000.0)
    if raw.min() < -32768 or raw.max() > 32767:
        raise InvalidRecordError("signal exceeds int16 at gain 1000")
    staged_dir.mkdir(parents=True, exist_ok=True)
    hea = staged_dir / f"{record_id}.hea"
    dat = staged_dir / f"{record_id}.dat"
    np.ascontiguousarray(raw.astype("<i2").T).tofile(str(dat))
    rec_fs = int(fs) if float(fs).is_integer() else fs
    lines = [f"{record_id} 12 {rec_fs} {n}"]
    for lead in LEADS:
        lines.append(f"{record_id}.dat 16 1000.0(0)/mV 16 0 0 0 0 {lead}")
    for key in ("Age", "Sex"):
        if comments.get(key):
            lines.append(f"# {key}: {comments[key]}")
    hea.write_text("\n".join(lines) + "\n")
    back = np.fromfile(str(dat), dtype="<i2",
                       count=12 * n).astype(np.int64).reshape(n, 12).T
    if back.shape != raw.shape or not np.array_equal(back, raw.astype(np.int64)):
        raise InvalidRecordError("staging round-trip mismatch")
    return hea


def run_kit(staged_dir, render_dir):
    """Invoke the kit batch driver once for a staged directory."""
    kit_dir = Path(CFG["kit_dir"])
    if not (kit_dir / KIT_SCRIPT).is_file():
        raise RuntimeError(f"kit script missing: {kit_dir / KIT_SCRIPT}")
    render_dir.mkdir(parents=True, exist_ok=True)
    # NOTE: absolute paths are mandatory — the kit subprocess runs with
    # cwd=kit_dir, so relative -i/-o would resolve inside the kit tree.
    cmd = [CFG["kit_python"], KIT_SCRIPT,
           "-i", str(Path(staged_dir).resolve()),
           "-o", str(Path(render_dir).resolve()),
           "-se", str(CFG["kit_seed"]), *CFG["kit_render_flags"]]
    env = dict(os.environ)
    env.update(CFG.get("kit_env", {}))
    env["ECG_FIXED_FONT"] = CFG["kit_fixed_font"]
    # one numerical thread per renderer; never the GPU
    env.update({"CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1",
                "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
    proc = subprocess.run(cmd, cwd=str(kit_dir), capture_output=True,
                          text=True, timeout=KIT_TIMEOUT_S, env=env)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-3:]
        raise RuntimeError(f"kit batch failed "
                           f"({staged_dir} -> {render_dir}): {'; '.join(tail)}")


def collect_pngs(render_dir, record_ids):
    """Map record_id -> rendered PNG. The kit names outputs <record>-0.png."""
    found = {}
    for png in Path(render_dir).rglob("*.png"):
        stem = png.stem  # <record_id>-0
        rid = stem[:-2] if stem.endswith("-0") else stem
        if rid in record_ids:
            found[rid] = png
    missing = sorted(set(record_ids) - set(found))
    if missing:
        raise RuntimeError(f"kit produced no PNG for {len(missing)} records, "
                           f"e.g. {missing[:5]}")
    return found


def clean_render_dir(render_dir):
    """Remove kit side products; only *.png may remain."""
    for pat in ("*.hea", "*.dat", "*.json", "*.csv"):
        for p in Path(render_dir).rglob(pat):
            try:
                p.unlink()
            except OSError:
                pass


def record_id_for(rel):
    stem = Path(rel).with_suffix("").as_posix()
    rid = stem.replace("/", "__").replace("\\", "__")
    if not rid or rid in (".", "..") or "/" in rid or len(rid) > 180:
        raise InvalidRecordError(f"Unusable record id for {rel!r}")
    return rid


def _base_row(rel):
    return {"source_header": rel,
            "image_path": str(Path("images") / Path(rel).with_suffix(".png")),
            "status": "", "error": "", "sample_rate_hz": "", "samples": "",
            "duration_s": "", "image_width": "", "image_height": "",
            "image_bytes": ""}


def process_batch(batch):
    scratch = (Path(CFG["kit_scratch_dir"])
               / f"w{os.getpid()}-{int(time.time() * 1000) % 100000000}")
    staged = scratch / "staged"
    render = scratch / "render"
    rows = {}
    todo = {}
    for rel in batch:
        base = _base_row(rel)
        rows[rel] = base
        header = INPUT / rel
        try:
            prows, names, fs, nsamp, duration, comments = parse_header(header)
            base.update(sample_rate_hz=fs, samples=nsamp,
                        duration_s=round(duration, 6))
            out = OUTPUT / base["image_path"]
            if out.is_file() and out.stat().st_size > 100:
                try:
                    with Image.open(out) as existing:
                        ok = existing.size == (FINAL_W, FINAL_H)
                        existing.verify()
                    if ok:
                        base.update(status="skipped", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                        continue
                except (OSError, ValueError):
                    pass
                # A mismatched or damaged existing image is regenerated.
            mv = load_mv(header, prows, names, nsamp)
            rid = record_id_for(rel)
            if rid in todo:
                raise InvalidRecordError(f"Duplicate staged id {rid}")
            todo[rid] = (rel, mv, fs, comments)
        except InvalidRecordError as exc:
            base.update(status="invalid", error=str(exc)[:1000])
        except Exception as exc:
            base.update(status="failed",
                        error=f"{type(exc).__name__}: {exc}"[:1000])
    try:
        if todo:
            for rid, (rel, mv, fs, comments) in todo.items():
                if rows[rel]["status"]:
                    continue
                try:
                    stage_record(rid, mv, fs, comments, staged)
                except InvalidRecordError as exc:
                    rows[rel].update(status="invalid", error=str(exc)[:1000])
            staged_ids = [rid for rid in todo if not rows[todo[rid][0]]["status"]]
            if staged_ids:
                try:
                    run_kit(staged, render)
                    got = collect_pngs(render, set(staged_ids))
                except Exception as exc:
                    for rid in staged_ids:
                        rows[todo[rid][0]].update(
                            status="failed",
                            error=f"{type(exc).__name__}: {exc}"[:1000])
                    got = {}
                for rid, png in got.items():
                    rel = todo[rid][0]
                    base = rows[rel]
                    try:
                        with Image.open(png) as master:
                            if master.size != (MASTER_W, MASTER_H):
                                raise InvalidRecordError(
                                    f"master size {master.size}, "
                                    f"need {(MASTER_W, MASTER_H)}")
                            view = recolour_clean(master)
                        final = to_final(view)
                        out = OUTPUT / base["image_path"]
                        out.parent.mkdir(parents=True, exist_ok=True)
                        temp = out.with_name(out.name + f".{os.getpid()}.part")
                        try:
                            final.save(temp, format="PNG",
                                       compress_level=CFG["png_compress_level"])
                            temp.replace(out)
                        finally:
                            try:
                                temp.unlink(missing_ok=True)
                            except OSError:
                                pass
                            final.close()
                        base.update(status="success", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                    except InvalidRecordError as exc:
                        base.update(status="invalid", error=str(exc)[:1000])
                    except Exception as exc:
                        base.update(status="failed",
                                    error=f"{type(exc).__name__}: {exc}"[:1000])
                clean_render_dir(render)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
    return [rows[rel] for rel in batch]


def process_one(rel):
    return process_batch([rel])[0]


def ping():
    """Zero-cost import probe: lets §7 verify a spawn-context pool child
    can import this module without rendering anything."""
    return {"ok": True, "leads": list(LEADS),
            "final": (FINAL_W, FINAL_H), "master": (MASTER_W, MASTER_H)}

'''

AUGMENT_WORKER_SOURCE = r'''"""Augmented clean12 worker: one paper/photo sheet per clean12 ECG record.

Each record is rendered by the clean12 render worker (ecg_clean_worker, the
module embedded in Clean12_ECG_Signal_Images_EC2.ipynb: canonical WFDB
staging + the patched ECG-image-kit with the clean12 flags) to its 2200x1700
grey kit master, exactly as for the clean12 sheets. The master then goes
through the image pipeline's stage-4 paper/photo augmentation
(dataset-combination-code/ecg-dataset-pipeline/image_pipeline/stage4_augment.py,
the code behind the correct_prediction_samples previews), a JPEG q85 round
trip, and the clean view's own 1024x768 fit (ecg_clean_worker.to_final).

The stage-4 functions below are a line-for-line port (checked pixel for
pixel against stage4_augment.augment with the stage-4 settings). Two changes:
wrinkled paper and fold creases are stronger than stage 4 (WRINKLE/CREASE
settings below, raised on request after comparing previews), and with a plan
the balanced factors (grid palette, ink strength, tilt, perspective, fold
creases, edge shadow, blur) are forced to the planned level. Every other
value keeps its stage-4 distribution.
"""
from __future__ import annotations

import hashlib
import io
import os
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter

import ecg_clean_worker as R

# ---------------------------------------------------------------- stage-4 config
# Values copied from image_pipeline/config.py (the corpus stage-4 settings),
# except the wrinkle/crease block marked STRONGER.
STAGE4_SEED = 20260818
GRID_PALETTES = {
    # major gridline        minor gridline        paper
    "red":    ((178, 86, 84), (228, 180, 176), (253, 249, 247)),
    "pink":   ((188, 124, 148), (236, 200, 212), (254, 251, 252)),
    "orange": ((186, 126, 86), (236, 202, 176), (254, 250, 245)),
    "grey":   ((124, 124, 124), (198, 198, 198), (252, 252, 252)),
}
GRID_PALETTE_WEIGHTS = {"red": 0.38, "orange": 0.22, "pink": 0.20, "grey": 0.20}
GRID_SATURATION = (0.72, 1.02)
GRID_TRACE_MAX = 55
GRID_MAJOR_LEVEL = 102
GRID_MINOR_LEVEL = 191
GRID_PAPER_LEVEL = 250
TILT_RANGE = (-3.0, 3.0)
KEYSTONE_PROB = 0.7
KEYSTONE_MAG = (0.004, 0.018)
# STRONGER than stage 4 (stage 4: texture 0.30-0.62 clip 0.86-1.10; crease
# amp 0.06-0.13 clip 0.88-1.14 width 0.006-0.020; bow 0.02-0.06). Wrinkles
# read clearly and folds show as sharp ridges; the darkest fold still sits far
# above the trace ink, so traces stay legible.
TEXTURE_STRENGTH = (0.65, 1.00)
TEXTURE_CLIP = (0.76, 1.14)
CREASE_PROB = 0.62               # stage-4 draw; the balanced plan sets creases on/off
CREASE_AMP = (0.16, 0.26)
CREASE_CLIP = (0.76, 1.22)
CREASE_WIDTH = (0.004, 0.012)   # fraction of the long side
BOW_AMP = (0.03, 0.08)
EDGE_SHADOW_PROB = 0.65
PAPER_LEVEL_TARGET = (216, 248)
DEFOCUS_PROB = 0.55
DEFOCUS_RADIUS = (0.3, 0.9)
MARGIN_PX = 18
JPEG_QUALITY = 85
JPEG_SUBSAMPLING = 2  # 4:2:0

# ---------------------------------------------------------------- balanced plan
AUG_PALETTES = ("orange", "pink", "grey")  # red is the clean view's grid
INK_BINS = ((0.72, 0.82), (0.82, 0.92), (0.92, 1.02))
TILT_BINS = ((-3.0, -2.0), (-2.0, -1.0), (-1.0, 0.0), (0.0, 1.0), (1.0, 2.0), (2.0, 3.0))
PLAN_FACTORS = {
    "grid_palette": AUG_PALETTES,
    "ink_bin": tuple(range(len(INK_BINS))),
    "tilt_bin": tuple(range(len(TILT_BINS))),
    "perspective": (0, 1),
    "fold_creases": (0, 1),
    "edge_shadow": (0, 1),
    "blur": (0, 1),
}

MANIFEST_FIELDS = [
    "record_path", "image_path", "status", "error", "image_bytes",
    "grid_palette", "ink_bin", "grid_saturation", "ink_level", "tilt_bin", "tilt_deg",
    "perspective", "perspective_mag", "wrinkle_texture", "wrinkle_strength",
    "fold_creases", "fold_lines", "paper_bows", "light_angle_deg", "light_ramp",
    "corner_falloff", "edge_shadow", "shadow_edge", "shadow_depth", "shadow_width",
    "tint_rgb_gain", "brightness", "contrast", "paper_level", "sensor_noise_sigma",
    "blur", "blur_radius_px",
]

TEXTURE_DIR: Path | None = None
_TEXTURES: tuple[Path, ...] | None = None


def set_texture_dir(path) -> None:
    global TEXTURE_DIR, _TEXTURES
    TEXTURE_DIR, _TEXTURES = Path(path), None


def _texture_files() -> tuple[Path, ...]:
    """The kit's wrinkled-paper photographs (CreasesWrinkles/wrinkles-dataset)."""
    global _TEXTURES
    if _TEXTURES is None:
        if TEXTURE_DIR is None or not TEXTURE_DIR.is_dir():
            raise RuntimeError(f"wrinkle texture folder missing: {TEXTURE_DIR}")
        _TEXTURES = tuple(sorted(p for p in TEXTURE_DIR.iterdir()
                                 if p.suffix.lower() in {".jpg", ".jpeg", ".png"}))
        if not _TEXTURES:
            raise RuntimeError(f"no wrinkle textures in {TEXTURE_DIR}")
    return _TEXTURES


def blake_int(*parts) -> int:
    key = "|".join(str(p) for p in parts).encode()
    return int.from_bytes(hashlib.blake2b(key, digest_size=8).digest(), "big")


def stage4_rng(seed_parts: tuple) -> np.random.Generator:
    """stage4_augment._rng: per-image generator keyed by output identity only."""
    return np.random.default_rng(blake_int(*seed_parts) ^ STAGE4_SEED)


# ---------------------------------------------------------------- stage-4 port
def _paper_colour(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    k = max(8, min(h, w) // 40)
    patches = np.concatenate([
        img[:k, :k].reshape(-1, 3), img[:k, -k:].reshape(-1, 3),
        img[-k:, :k].reshape(-1, 3), img[-k:, -k:].reshape(-1, 3)])
    return np.median(patches, axis=0)


def _rotate_on_paper(img: Image.Image, deg: float, fill) -> Image.Image:
    w, h = img.size
    big = img.rotate(deg, resample=Image.BICUBIC, expand=True, fillcolor=fill)
    bw, bh = big.size
    left, top = (bw - w) // 2, (bh - h) // 2
    return big.crop((left, top, left + w, top + h))


def _illumination(arr, rng, meta, shadow=None):
    h, w = arr.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    ang = rng.uniform(0, 2 * np.pi)
    ramp = (np.cos(ang) * (xx / w - 0.5) + np.sin(ang) * (yy / h - 0.5))
    ramp_amp = rng.uniform(0.10, 0.22)
    field = 1.0 + ramp_amp * ramp * 2.0
    r = np.sqrt((xx / w - 0.5) ** 2 + (yy / h - 0.5) ** 2)
    falloff = rng.uniform(0.06, 0.18)
    field *= 1.0 - falloff * (r / r.max()) ** 2
    drawn = rng.random() < EDGE_SHADOW_PROB
    on = drawn if shadow is None else bool(shadow)
    meta.update(light_angle_deg=round(float(np.degrees(ang)), 1),
                light_ramp=round(float(ramp_amp), 4), corner_falloff=round(float(falloff), 4),
                edge_shadow=int(on), shadow_edge="", shadow_depth="", shadow_width="")
    if on:
        edge = int(rng.integers(0, 4))
        d = {0: xx / w, 1: 1 - xx / w, 2: yy / h, 3: 1 - yy / h}[edge]
        depth, width = rng.uniform(0.10, 0.26), rng.uniform(0.04, 0.12)
        field *= 1.0 - depth * np.exp(-d / width)
        meta.update(shadow_edge=("left", "right", "top", "bottom")[edge],
                    shadow_depth=round(float(depth), 4), shadow_width=round(float(width), 4))
    return arr * field[..., None]


def _paper_texture(arr, rng, meta):
    files = _texture_files()
    h, w = arr.shape[:2]
    pick = files[int(rng.integers(len(files)))]
    tex = Image.open(pick).convert("L")
    tw, th = tex.size
    scale = rng.uniform(0.55, 1.0)
    cw, ch = max(16, int(tw * scale)), max(16, int(th * scale))
    x0 = int(rng.integers(0, max(1, tw - cw + 1)))
    y0 = int(rng.integers(0, max(1, th - ch + 1)))
    tex = tex.crop((x0, y0, x0 + cw, y0 + ch))
    if rng.random() < 0.5:
        tex = tex.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.5:
        tex = tex.transpose(Image.FLIP_TOP_BOTTOM)
    if rng.random() < 0.5:
        tex = tex.transpose(Image.ROTATE_90)
    field = np.asarray(tex.resize((w, h), Image.LANCZOS), dtype=np.float32) / 255.0
    mean = float(field.mean())
    if mean <= 1e-3:
        raise RuntimeError(f"blank wrinkle texture {pick.name}")
    strength = rng.uniform(*TEXTURE_STRENGTH)
    meta.update(wrinkle_texture=pick.name, wrinkle_strength=round(float(strength), 4))
    field = 1.0 + (field / mean - 1.0) * strength
    return arr * np.clip(field, *TEXTURE_CLIP)[..., None]


def _creases(arr, rng, meta):
    h, w = arr.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    field = np.ones((h, w), dtype=np.float32)
    folds = []
    for _ in range(int(rng.integers(1, 4))):
        if rng.random() < 0.5:
            pos = rng.uniform(0.12, 0.88) * h
            tilt = rng.uniform(-0.06, 0.06)
            dist = yy - (pos + tilt * (xx - w / 2))
            folds.append("h")
        else:
            pos = rng.uniform(0.12, 0.88) * w
            tilt = rng.uniform(-0.06, 0.06)
            dist = xx - (pos + tilt * (yy - h / 2))
            folds.append("v")
        width = rng.uniform(*CREASE_WIDTH) * max(h, w)
        amp = rng.uniform(*CREASE_AMP)
        g = np.exp(-(dist ** 2) / (2 * width ** 2))
        field *= 1.0 + amp * g * (dist / width) / 1.6
    bows = int(rng.integers(1, 3))
    for _ in range(bows):
        k = rng.uniform(0.6, 1.8)
        phase = rng.uniform(0, 2 * np.pi)
        axis = yy / h if rng.random() < 0.5 else xx / w
        field *= 1.0 + rng.uniform(*BOW_AMP) * np.sin(2 * np.pi * k * axis + phase)
    meta.update(fold_lines="".join(folds), paper_bows=bows)
    return arr * np.clip(field, *CREASE_CLIP)[..., None]


def _normalise_paper(arr, rng, meta):
    lum = arr @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    paper = float(np.percentile(lum, 95))
    if paper <= 1.0:
        return arr
    target = rng.uniform(*PAPER_LEVEL_TARGET)
    meta["paper_level"] = round(float(target), 2)
    return arr * (target / paper)


def _colour_cast(arr, rng, meta):
    gains = 1.0 + rng.normal(0.0, 0.012, size=3)
    meta["tint_rgb_gain"] = "/".join(f"{g:.4f}" for g in gains)
    return arr * gains[None, None, :]


def _grid_lut(palette, rng, sat_bin=None):
    major, minor, paper = (np.asarray(c, dtype=np.float32) for c in palette)
    ink = np.full(3, rng.uniform(8, 34), dtype=np.float32)
    ink += rng.normal(0, 3, size=3)
    paper = paper * rng.uniform(0.985, 1.005) + rng.normal(0, 1.2, size=3)
    sat = rng.uniform(*GRID_SATURATION)
    if sat_bin is not None:  # same draw, mapped into the planned bin
        lo, hi = INK_BINS[sat_bin]
        u = (sat - GRID_SATURATION[0]) / (GRID_SATURATION[1] - GRID_SATURATION[0])
        sat = lo + u * (hi - lo)

    def _sat(c):
        lum = float(np.dot(c, (0.299, 0.587, 0.114)))
        return np.clip(lum + (c - lum) * sat, 0, 255)

    major, minor = _sat(major), _sat(minor)
    xs = np.array([0, GRID_TRACE_MAX, GRID_MAJOR_LEVEL, GRID_MINOR_LEVEL,
                   GRID_PAPER_LEVEL, 255], dtype=np.float32)
    lut = np.empty((256, 3), dtype=np.float32)
    grid = np.arange(256, dtype=np.float32)
    for ch in range(3):
        ys = np.array([ink[ch] * 0.35, ink[ch], major[ch], minor[ch], paper[ch],
                       min(255.0, paper[ch] + 3)], dtype=np.float32)
        lut[:, ch] = np.interp(grid, xs, ys)
    return np.clip(lut, 0, 255), float(sat), float(ink.mean())


def _recolour_grid(master: Image.Image, rng, force=None, sat_bin=None):
    names = list(GRID_PALETTES)
    weights = np.array([GRID_PALETTE_WEIGHTS.get(n, 0.0) for n in names], dtype=np.float64)
    weights = weights / weights.sum()
    drawn = names[int(rng.choice(len(names), p=weights))]  # draw, THEN override (stage 4)
    name = force if force in GRID_PALETTES else drawn
    lut, sat, ink = _grid_lut(GRID_PALETTES[name], rng, sat_bin)
    out = Image.fromarray(lut[np.asarray(master.convert("L"))].astype(np.uint8), "RGB")
    return out, name, sat, ink


def _defocus(img, rng, meta, blur=None):
    drawn = rng.random() < DEFOCUS_PROB
    on = drawn if blur is None else bool(blur)
    meta.update(blur=int(on), blur_radius_px="")
    if not on:
        return img
    radius = float(rng.uniform(*DEFOCUS_RADIUS))
    meta["blur_radius_px"] = round(radius, 4)
    return img.filter(ImageFilter.GaussianBlur(radius=radius))


def augment(master: Image.Image, rng: np.random.Generator, plan: dict | None = None):
    """stage4_augment.augment on a grey kit master (RGB), in memory.

    Returns (RGB sheet with the stage-4 margin, effect metadata). plan keys are
    PLAN_FACTORS; plan=None reproduces stage 4 draw for draw.
    """
    plan = plan or {}
    meta: dict = {}
    img, palette, sat, ink = _recolour_grid(master, rng, plan.get("grid_palette"), plan.get("ink_bin"))
    meta.update(grid_palette=palette, grid_saturation=round(sat, 4), ink_level=round(ink, 2))
    arr0 = np.asarray(img, dtype=np.float32)
    fill = tuple(int(v) for v in _paper_colour(arr0))

    deg = float(rng.uniform(*TILT_RANGE))
    if "tilt_bin" in plan:
        lo, hi = TILT_BINS[plan["tilt_bin"]]
        deg = lo + (deg - TILT_RANGE[0]) / (TILT_RANGE[1] - TILT_RANGE[0]) * (hi - lo)
    meta["tilt_deg"] = round(deg, 3)
    img = _rotate_on_paper(img, deg, fill)

    keystone = rng.random() < KEYSTONE_PROB
    if "perspective" in plan:
        keystone = bool(plan["perspective"])
    meta.update(perspective=int(keystone), perspective_mag="")
    if keystone:
        w, h = img.size
        m = rng.uniform(*KEYSTONE_MAG)
        dx, dy = m * w, m * h
        quad = (rng.uniform(0, dx), rng.uniform(0, dy),
                rng.uniform(0, dx), h - rng.uniform(0, dy),
                w - rng.uniform(0, dx), h - rng.uniform(0, dy),
                w - rng.uniform(0, dx), rng.uniform(0, dy))
        meta["perspective_mag"] = round(float(m), 5)
        img = img.transform((w, h), Image.QUAD, quad, resample=Image.BICUBIC, fillcolor=fill)

    arr = np.asarray(img, dtype=np.float32)
    arr = _paper_texture(arr, rng, meta)
    creases = rng.random() < CREASE_PROB
    if "fold_creases" in plan:
        creases = bool(plan["fold_creases"])
    meta.update(fold_creases=int(creases), fold_lines="", paper_bows="")
    if creases:
        arr = _creases(arr, rng, meta)
    arr = _illumination(arr, rng, meta, plan.get("edge_shadow"))
    arr = _colour_cast(arr, rng, meta)
    exposure = rng.uniform(0.92, 1.06)
    arr *= exposure
    mean = arr.mean()
    contrast = rng.uniform(0.94, 1.08)
    arr = (arr - mean) * contrast + mean
    meta.update(brightness=round(float(exposure), 4), contrast=round(float(contrast), 4))
    arr = _normalise_paper(arr, rng, meta)
    noise = rng.uniform(1.5, 4.5)
    meta["sensor_noise_sigma"] = round(float(noise), 3)
    arr += rng.normal(0.0, noise, size=arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    out = _defocus(Image.fromarray(arr), rng, meta, plan.get("blur"))

    m = MARGIN_PX
    canvas = Image.new("RGB", (out.width + 2 * m, out.height + 2 * m), fill)
    canvas.paste(out, (m, m))
    return canvas, meta


def jpeg_roundtrip(img: Image.Image) -> Image.Image:
    """Stage 4 writes its photograph as JPEG q85 4:2:0; keep those artifacts."""
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=JPEG_QUALITY, subsampling=JPEG_SUBSAMPLING)
    buf.seek(0)
    out = Image.open(buf)
    out.load()
    return out


def augment_master(master: Image.Image, seed: int, record_path: str, plan: dict):
    """Grey kit master -> final 1024x768 augmented sheet + metadata."""
    sheet, meta = augment(master.convert("RGB"), stage4_rng((seed, record_path, "clean12-aug")), plan)
    return R.to_final(jpeg_roundtrip(sheet)), meta


def check_plan(meta: dict, plan: dict) -> None:
    """Raise unless the rendered effects are exactly the planned levels."""
    for factor in ("grid_palette", "perspective", "fold_creases", "edge_shadow", "blur"):
        if meta[factor] != plan[factor]:
            raise RuntimeError(f"{factor}: planned {plan[factor]!r}, rendered {meta[factor]!r}")
    lo, hi = TILT_BINS[plan["tilt_bin"]]
    if not lo - 1e-3 <= meta["tilt_deg"] <= hi + 1e-3:
        raise RuntimeError(f"tilt {meta['tilt_deg']} outside planned bin {lo}..{hi}")
    lo, hi = INK_BINS[plan["ink_bin"]]
    if not lo - 1e-3 <= meta["grid_saturation"] <= hi + 1e-3:
        raise RuntimeError(f"ink strength {meta['grid_saturation']} outside planned bin {lo}..{hi}")


# ---------------------------------------------------------------- balanced plan
def balanced_plan(records: list[dict], seed: int) -> dict[str, dict]:
    """Assign every factor level equally, independently of the labels.

    records: dicts with record_path, site, labels. For each factor the records
    are ordered by stratum (site, label set) and, inside a stratum, by a
    per-factor seeded hash; levels are dealt round-robin along that order with
    one counter that runs across strata. So every factor is exactly balanced
    overall (counts differ by at most 1), balanced to +-1 inside every label
    combination, and paired with the other factors by independent shuffles.
    """
    keyed = sorted(records, key=lambda r: (r["site"], r["labels"], r["record_path"]))
    plan = {r["record_path"]: {} for r in keyed}
    for factor, levels in PLAN_FACTORS.items():
        order = sorted(keyed, key=lambda r: (r["site"], r["labels"],
                                            blake_int(seed, factor, r["record_path"])))
        offset = blake_int(seed, factor, "offset") % len(levels)
        for i, r in enumerate(order):
            plan[r["record_path"]][factor] = levels[(i + offset) % len(levels)]
    return plan


# ---------------------------------------------------------------- pool worker
_SEED: int | None = None


def init_worker(config: dict) -> None:
    """config: the clean12 render_config plus seed, texture_dir, input_dir, output_dir."""
    global _SEED
    for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[name] = "1"
    R.init_worker(config, config["input_dir"], config["output_dir"])
    set_texture_dir(config["texture_dir"])
    _texture_files()
    _SEED = int(config["seed"])


def process_batch(jobs: list[dict]) -> list[dict]:
    """jobs: record_path, image_path, plan. One kit invocation renders the whole
    batch to grey masters (scratch, deleted afterwards); each master is then
    augmented and written to OUTPUT/image_path (temp file + atomic rename)."""
    scratch = Path(R.CFG["kit_scratch_dir"]) / f"a{os.getpid()}-{int(time.time() * 1000) % 100000000}"
    staged, render = scratch / "staged", scratch / "render"
    rows, todo = {}, {}
    for job in jobs:
        rel = job["record_path"]
        rows[rel] = {"record_path": rel, "image_path": job["image_path"], "status": "", "error": ""}
        try:
            header = R.INPUT / rel
            prows, names, fs, nsamp, _, comments = R.parse_header(header)
            mv = R.load_mv(header, prows, names, nsamp)
            rid = R.record_id_for(rel)
            if rid in todo:
                raise RuntimeError(f"duplicate staged id {rid}")
            R.stage_record(rid, mv, fs, comments, staged)
            todo[rid] = job
        except Exception as exc:  # noqa: BLE001 - one bad record never stops the batch
            rows[rel].update(status="failed", error=f"{type(exc).__name__}: {exc}"[:1000])
    try:
        if todo:
            try:
                R.run_kit(staged, render)
                masters = R.collect_pngs(render, set(todo))
            except Exception as exc:  # noqa: BLE001
                masters = {}
                for job in todo.values():
                    rows[job["record_path"]].update(status="failed",
                                                    error=f"{type(exc).__name__}: {exc}"[:1000])
            for rid, png in masters.items():
                job = todo[rid]
                row = rows[job["record_path"]]
                try:
                    with Image.open(png) as master:
                        if master.size != (R.MASTER_W, R.MASTER_H):
                            raise RuntimeError(f"master size {master.size}")
                        final, meta = augment_master(master, _SEED, job["record_path"], job["plan"])
                    check_plan(meta, job["plan"])
                    out = R.OUTPUT / job["image_path"]
                    out.parent.mkdir(parents=True, exist_ok=True)
                    temp = out.with_name(out.name + f".{os.getpid()}.part")
                    final.save(temp, "PNG", compress_level=R.CFG["png_compress_level"])
                    os.replace(temp, out)
                    row.update(meta, status="success", image_bytes=out.stat().st_size,
                               ink_bin=job["plan"]["ink_bin"], tilt_bin=job["plan"]["tilt_bin"])
                except Exception as exc:  # noqa: BLE001
                    row.update(status="failed", error=f"{type(exc).__name__}: {exc}"[:1000])
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
    return [rows[job["record_path"]] for job in jobs]
'''

for _name, _source in (("ecg_clean_worker.py", CLEAN_WORKER_SOURCE), ("augment_worker.py", AUGMENT_WORKER_SOURCE)):
    _path = WORK_DIR / _name
    compile(_source, str(_path), "exec")
    if not _path.is_file() or _path.read_text() != _source:
        _path.write_text(_source)
WORKER_SHA256 = hashlib.sha256((CLEAN_WORKER_SOURCE + AUGMENT_WORKER_SOURCE).encode()).hexdigest()
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(WORK_DIR), os.environ.get("PYTHONPATH")]))
importlib.invalidate_caches()
import ecg_clean_worker
import augment_worker as AW
ecg_clean_worker = importlib.reload(ecg_clean_worker)
AW = importlib.reload(AW)

render_config = {
    "kit_dir": KIT_DIR, "kit_python": sys.executable, "kit_seed": KIT_SEED,
    "kit_fixed_font": KIT_FIXED_FONT, "kit_render_flags": KIT_RENDER_FLAGS, "kit_env": KIT_ENV,
    "kit_scratch_dir": str(KIT_SCRATCH_DIR), "max_duration_s": MAX_DURATION_S,
    "png_compress_level": PNG_COMPRESS_LEVEL,
}
WORKER_CONFIG = {**render_config, "seed": AUGMENT_SEED, "texture_dir": str(TEXTURE_DIR),
                 "input_dir": str(INPUT_DIR), "output_dir": str(OUTPUT_DIR)}
KIT_SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
print("Workers written; SHA-256", WORKER_SHA256[:16], "| palettes", AW.AUG_PALETTES)
STAGES_DONE.add("workers")

## 7. Balanced plan, run configuration and balance audit

`AW.balanced_plan` assigns every record one level of each balanced factor. It is deterministic for a given `AUGMENT_SEED`, so a rerun reproduces it exactly. This cell writes into the output folder, which is packaged with the images:
- `augmentation_plan.csv`;
- `balance_by_condition.csv` (count of each condition × factor × level);
- `record_manifest.csv` (the clean12 labels, for joins);
- `config.json`.

It stops unless every factor is balanced to ±1 overall and inside every `(site, label set)` stratum. A config lock refuses to mix images made with different settings in one folder. It also defines the pool runner used by §8, §8b and §9, and loads the progress log so that already-rendered sheets are not rendered again.

In [ ]:
plan = AW.balanced_plan(records, AUGMENT_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)

run_config = {
    "dataset": "HEEDB clean12, augmented paper/photo sheet (one per record)",
    "source_uri": S3_SOURCE, "source_sha256": expected_sha256, "source_etag": source_identity["etag"],
    "record_count": len(records), "seed": AUGMENT_SEED, "worker_sha256": WORKER_SHA256,
    "kit_bundle_sha256": KIT_BUNDLE_SHA256, "render": {k: v for k, v in render_config.items()
                                                       if k not in ("kit_dir", "kit_python", "kit_scratch_dir")},
    "augmentation": "kit grey master (clean12 settings) -> stage4_augment.augment with the balanced "
                    "plan -> JPEG q85 4:2:0 round trip -> clean12 to_final (1024x768)",
    "balanced_factors": {k: list(v) for k, v in AW.PLAN_FACTORS.items()},
    "ink_bins": AW.INK_BINS, "tilt_bins_deg": AW.TILT_BINS,
    "grid_palettes": {k: AW.GRID_PALETTES[k] for k in AW.AUG_PALETTES},
    "always_on": {"wrinkle_texture_strength": AW.TEXTURE_STRENGTH, "wrinkle_clip": AW.TEXTURE_CLIP,
                  "light_ramp": [0.10, 0.22], "corner_falloff": [0.06, 0.18], "tint_sigma": 0.012,
                  "brightness": [0.92, 1.06], "contrast": [0.94, 1.08], "paper_level": AW.PAPER_LEVEL_TARGET,
                  "sensor_noise_sigma": [1.5, 4.5], "jpeg_quality": AW.JPEG_QUALITY},
    "when_on": {"perspective_mag": AW.KEYSTONE_MAG, "crease_amplitude": AW.CREASE_AMP,
                "crease_clip": AW.CREASE_CLIP, "crease_width": AW.CREASE_WIDTH, "paper_bow": AW.BOW_AMP,
                "edge_shadow_depth": [0.10, 0.26], "blur_radius_px": AW.DEFOCUS_RADIUS},
}
config_hash = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()
lock_file = OUTPUT_DIR / ".augment_config_sha256"
if lock_file.is_file() and lock_file.read_text().strip() != config_hash:
    raise RuntimeError("OUTPUT_DIR holds images made with different settings. Use a fresh OUTPUT_DIR "
                       "(and ARCHIVE_BASENAME) or restore the previous settings.")
lock_file.write_text(config_hash + "\n")
(OUTPUT_DIR / "config.json").write_text(json.dumps(run_config, indent=2) + "\n")
shutil.copy2(INPUT_DIR / "manifest.csv", OUTPUT_DIR / "record_manifest.csv")

factors = list(AW.PLAN_FACTORS)
with (OUTPUT_DIR / "augmentation_plan.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["record_path", "image_path", "site", "labels", *factors])
    writer.writeheader()
    for r in records:
        writer.writerow({"record_path": r["record_path"], "image_path": r["image_path"],
                         "site": r["site"], "labels": r["labels"], **plan[r["record_path"]]})

audit_failures = []
for factor, levels in AW.PLAN_FACTORS.items():
    counts = Counter(plan[r["record_path"]][factor] for r in records)
    if set(counts) != set(levels) or max(counts.values()) - min(counts.values()) > 1:
        audit_failures.append(f"{factor}: overall counts {dict(counts)}")
strata = defaultdict(list)
for r in records:
    strata[(r["site"], r["labels"])].append(r["record_path"])
for key, paths in strata.items():
    for factor, levels in AW.PLAN_FACTORS.items():
        counts = Counter(plan[p][factor] for p in paths)
        if max(counts.get(l, 0) for l in levels) - min(counts.get(l, 0) for l in levels) > 1:
            audit_failures.append(f"{factor} in stratum {key}: {dict(counts)}")
if audit_failures:
    raise RuntimeError("Plan is not balanced:\n- " + "\n- ".join(audit_failures[:20]))

audit_rows, worst = [], {}
for condition in CONDITIONS:
    paths = [r["record_path"] for r in records if int(record_by_path[r["record_path"]][condition]) == 1]
    if not paths:
        continue
    for factor, levels in AW.PLAN_FACTORS.items():
        counts = Counter(plan[p][factor] for p in paths)
        for level in levels:
            audit_rows.append({"condition": condition, "records": len(paths), "factor": factor, "level": level,
                               "count": counts.get(level, 0), "share": round(counts.get(level, 0) / len(paths), 4)})
        spread = max(counts.get(l, 0) for l in levels) - min(counts.get(l, 0) for l in levels)
        worst[factor] = max(worst.get(factor, 0.0), spread / (len(paths) / len(levels)))
pd.DataFrame(audit_rows).to_csv(OUTPUT_DIR / "balance_by_condition.csv", index=False)
print(f"Plan: {len(records):,} records, balanced to +-1 overall and in all {len(strata)} (site, label set) strata.")
print("Worst per-condition spread (max-min)/expected:", {f: f"{100 * v:.1f}%" for f, v in worst.items()})
display(pd.DataFrame({f: pd.Series(Counter(str(plan[r["record_path"]][f]) for r in records)) for f in factors})
        .T.fillna("").astype(str))

def make_job(r):
    return {"record_path": r["record_path"], "image_path": r["image_path"], "plan": plan[r["record_path"]]}

def run_pool(subset, config, on_row, label):
    """Render subset in BATCH_SIZE tasks on WORKERS spawned processes; on_row(row) per record."""
    batches = iter([subset[i:i + BATCH_SIZE] for i in range(0, len(subset), BATCH_SIZE)])
    finished_n, last_report, t0 = 0, 0, time.monotonic()
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("spawn"),
                             initializer=AW.init_worker, initargs=(config,)) as pool:
        pending = {}

        def submit_next():
            batch = next(batches, None)
            if batch is not None:
                pending[pool.submit(AW.process_batch, [make_job(r) for r in batch])] = batch

        for _ in range(MAX_PENDING_BATCHES):
            submit_next()
        while pending:
            finished, _ = wait(pending, return_when=FIRST_COMPLETED)
            for future in finished:
                batch = pending.pop(future)
                try:
                    rows = future.result()
                except Exception as exc:   # task raised outside its per-record handling
                    rows = [{"record_path": r["record_path"], "image_path": r["image_path"], "status": "failed",
                             "error": f"task failure: {type(exc).__name__}: {exc}"[:1000]} for r in batch]
                for row in rows:
                    on_row(row)
                finished_n += len(rows)
                submit_next()
            if finished_n - last_report >= PROGRESS_EVERY or finished_n == len(subset):
                rate = finished_n / max(time.monotonic() - t0, 1e-9)
                print(f"{label}: {finished_n:,}/{len(subset):,} | {rate:.2f} records/s | "
                      f"ETA {(len(subset) - finished_n) / rate / 3600:.1f} h", flush=True)
                last_report = finished_n
    return time.monotonic() - t0

progress_path = WORK_DIR / "generation_progress.csv"
done, failed = {}, {}
if progress_path.is_file():
    with progress_path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            if row["status"] == "success" and (OUTPUT_DIR / row["image_path"]).is_file():
                done[row["record_path"]] = row
                failed.pop(row["record_path"], None)
            elif row["status"] != "success":
                done.pop(row["record_path"], None)
                failed[row["record_path"]] = row

def render_and_log(subset, label):
    """run_pool into OUTPUT_DIR, appending every row to the progress log."""
    with progress_path.open("a", newline="") as progress_file:
        writer = csv.DictWriter(progress_file, fieldnames=AW.MANIFEST_FIELDS, restval="")
        if progress_file.tell() == 0:
            writer.writeheader()

        def on_row(row):
            writer.writerow(row)
            progress_file.flush()
            if row["status"] == "success":
                done[row["record_path"]] = row
                failed.pop(row["record_path"], None)
            else:
                failed[row["record_path"]] = row

        return run_pool(subset, WORKER_CONFIG, on_row, label)

print(f"Progress log: {len(done):,} sheets already rendered")
STAGES_DONE.add("plan")

## 8. Preflight (run before the long job)

Takes about a minute. The run continues only if all of these pass:
- **instance:** identity from IMDS, with a warning if this is not a c8a.4xlarge;
- **auto-terminate:** an `ec2:TerminateInstances` **dry run** (only when `AUTO_TERMINATE`);
- **upload:** S3 write, read and delete permission at `S3_UPLOAD_PREFIX`;
- **disk** and **RAM** headroom;
- **one real record:** rendered by the kit in a spawned worker, augmented, and checked to be 1024x768 with its planned levels. The sheet goes to a scratch folder, which is deleted afterwards.

In [ ]:
pf_failures, pf_warnings = [], []
def pf_check(name, fn):
    try:
        print(f"  ok    {name}: {fn()}", flush=True)
    except Exception as exc:
        pf_failures.append(f"{name}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {name}: {type(exc).__name__}: {exc}", flush=True)

instance = None
def _pf_instance():
    global instance
    instance = ec2_identity()
    if instance["instance_type"] != "c8a.4xlarge":
        pf_warnings.append(f"instance type is {instance['instance_type']}, not c8a.4xlarge (timings assume it)")
    return instance
pf_check("EC2 instance identity (IMDS)", _pf_instance)

def _pf_terminate():
    if not AUTO_TERMINATE:
        return "AUTO_TERMINATE is off (skipped)"
    assert instance is not None, "no instance identity"
    ec2 = aws_session.client("ec2", region_name=instance["region"])
    try:
        ec2.terminate_instances(InstanceIds=[instance["instance_id"]], DryRun=True)
    except ClientError as exc:
        if exc.response["Error"]["Code"] == "DryRunOperation":
            return f"ec2:TerminateInstances allowed on {instance['instance_id']} (dry run)"
        raise RuntimeError(f"{exc.response['Error']['Code']}: give the instance role ec2:TerminateInstances "
                           "(top cell), or set AUTO_TERMINATE = False") from exc
    raise RuntimeError("dry run unexpectedly succeeded")
pf_check("auto-terminate permission", _pf_terminate)

def _pf_upload():
    up = urlparse(S3_UPLOAD_PREFIX)
    key = "/".join(p for p in (up.path.strip("/"), ".preflight_write_probe.txt") if p)
    body = f"preflight {datetime.now(timezone.utc).isoformat()}\n".encode()
    s3.put_object(Bucket=up.netloc, Key=key, Body=body)
    try:
        assert int(s3.head_object(Bucket=up.netloc, Key=key)["ContentLength"]) == len(body)
    finally:
        s3.delete_object(Bucket=up.netloc, Key=key)
    return f"write+read+delete OK at s3://{up.netloc}/{key}"
pf_check("S3 upload permission", _pf_upload)

def _pf_disk():
    remaining = len(records) - len(done)
    need = remaining * EST_IMAGE_MIB * 2**20 + 5 * 2**30
    free = shutil.disk_usage(OUTPUT_DIR).free
    assert free >= need, f"{free / 2**30:.0f} GiB free, need {need / 2**30:.0f} GiB for {remaining:,} sheets"
    return f"{free / 2**30:.0f} GiB free, {need / 2**30:.0f} GiB needed for {remaining:,} sheets"
pf_check("disk space", _pf_disk)

def _pf_ram():
    total = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    need = WORKERS * 0.7 + 3   # kit ~0.25 GiB, then augment ~0.6 GiB peak per worker; + parent
    assert total >= need, f"{total:.0f} GiB RAM < {need:.0f} GiB for {WORKERS} workers; lower WORKERS"
    return f"{total:.0f} GiB RAM for {WORKERS} workers"
pf_check("RAM", _pf_ram)

def _pf_one_record():
    probe_dir = WORK_DIR / "preflight"
    shutil.rmtree(probe_dir, ignore_errors=True)
    rows = []
    seconds = run_pool(records[:1], {**WORKER_CONFIG, "output_dir": str(probe_dir)}, rows.append, "preflight")
    row = rows[0]
    assert row["status"] == "success", row["error"]
    with Image.open(probe_dir / row["image_path"]) as image:
        assert image.size == (1024, 768) and image.mode == "RGB", (image.size, image.mode)
    shutil.rmtree(probe_dir)
    return (f"{row['record_path']} -> {row['grid_palette']}, tilt {row['tilt_deg']:+.2f}, creases "
            f"{row['fold_creases']}, {int(row['image_bytes']) / 2**20:.2f} MiB in {seconds:.1f} s incl. spawn + kit start")
pf_check("one record end to end (kit + augmentation, spawned worker)", _pf_one_record)

for warning in pf_warnings:
    print("  WARN ", warning)
if pf_failures:
    raise RuntimeError("PREFLIGHT FAIL; fix and rerun this cell:\n- " + "\n- ".join(pf_failures))
print("PREFLIGHT PASS")
STAGES_DONE.add("preflight")

## 8b. Throughput benchmark and time estimate (about 5 min)

Renders one full wave, `WORKERS × BATCH_SIZE` records, into the real output folder and logs them, so nothing is wasted. It then estimates the time for the rest. Measured on a 4-core laptop: about 4 s per record per core (kit about 2.6 s, augmentation and PNG about 1.4 s). On the c8a.4xlarge's 16 cores, expect roughly 7–9 h for 113,358 records.

In [ ]:
bench_subset = [r for r in records if r["record_path"] not in done][: WORKERS * BATCH_SIZE]
if bench_subset:
    bench_seconds = render_and_log(bench_subset, "benchmark")
    bench_rows = [done[r["record_path"]] for r in bench_subset if r["record_path"] in done]
    if len(bench_rows) != len(bench_subset):
        raise RuntimeError(f"{len(bench_subset) - len(bench_rows)} benchmark records failed: "
                           f"{[failed[r['record_path']]['error'] for r in bench_subset if r['record_path'] in failed][:3]}")
    rate = len(bench_rows) / bench_seconds
    mean_mib = np.mean([int(row["image_bytes"]) for row in bench_rows]) / 2**20
    remaining = len(records) - len(done)
    print(f"{rate:.2f} records/s with {WORKERS} workers ({WORKERS / rate:.2f} s per record per worker, "
          f"incl. pool start); mean PNG {mean_mib:.2f} MiB")
    print(f"Estimated time for the remaining {remaining:,} records: {remaining / rate / 3600:.1f} h; "
          f"output ~{len(records) * mean_mib / 1024:.0f} GiB in total")
else:
    print("Nothing left to benchmark: every record is already rendered.")
STAGES_DONE.add("benchmark")

## 9. Generate one augmented sheet per record

Records missing from the progress log are rendered. Failed records are retried `RETRY_PASSES` times. The cell raises unless:
- every record has its sheet (113,358 images for 113,358 records);
- the rendered effect counts equal the plan.

If it raises, nothing is packaged or uploaded, and the instance is not terminated.

In [ ]:
for stray in (OUTPUT_DIR / "images").rglob("*.part"):   # temp file of a killed worker
    stray.unlink()
started = time.monotonic()
for attempt in range(1 + RETRY_PASSES):
    todo = [r for r in records if r["record_path"] not in done]
    if not todo:
        break
    print(f"{'Retry pass ' + str(attempt) if attempt else 'Render'}: {len(todo):,} records", flush=True)
    render_and_log(todo, f"retry {attempt}" if attempt else "render")

still_failed = [failed.get(r["record_path"], {"record_path": r["record_path"], "error": "not rendered"})
                for r in records if r["record_path"] not in done]
with (OUTPUT_DIR / "failures.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=AW.MANIFEST_FIELDS, restval="")
    writer.writeheader()
    writer.writerows(still_failed)
if still_failed:
    raise RuntimeError(f"{len(still_failed):,} records failed after {RETRY_PASSES} retry pass(es); see "
                       f"{OUTPUT_DIR / 'failures.csv'}. Not packaged, not uploaded, instance left running.")

manifest_rows = [done[r["record_path"]] for r in records]
on_disk = sum(1 for _ in (OUTPUT_DIR / "images").rglob("*.png"))
missing = [row["image_path"] for row in manifest_rows if not (OUTPUT_DIR / row["image_path"]).is_file()]
if missing or on_disk != EXPECTED_RECORDS:
    raise RuntimeError(f"{len(missing)} sheets missing; {on_disk:,} PNGs on disk, expected {EXPECTED_RECORDS:,}")
effect_counts = {}
for factor in AW.PLAN_FACTORS:
    rendered_counts = Counter(str(row[factor]) for row in manifest_rows)
    planned_counts = Counter(str(plan[r["record_path"]][factor]) for r in records)
    if rendered_counts != planned_counts:
        raise RuntimeError(f"{factor}: rendered {dict(rendered_counts)} != planned {dict(planned_counts)}")
    effect_counts[factor] = dict(sorted(rendered_counts.items()))
(OUTPUT_DIR / "effect_counts.json").write_text(json.dumps(effect_counts, indent=2) + "\n")
with (OUTPUT_DIR / "manifest.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=AW.MANIFEST_FIELDS, restval="")
    writer.writeheader()
    writer.writerows(manifest_rows)
summary = {
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "source": S3_SOURCE, "source_sha256": expected_sha256,
    "records": len(records), "images": on_disk, "failed": 0,
    "image_bytes": sum(int(row["image_bytes"]) for row in manifest_rows),
    "generate_cell_seconds": round(time.monotonic() - started, 1), "config_sha256": config_hash,
    "effect_counts": effect_counts,
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps({k: v for k, v in summary.items() if k != "effect_counts"}, indent=2))
print("Effect counts:", effect_counts)
STAGES_DONE.add("generate")

## 10. Package and upload to S3 (one streaming pass)

The output folder is streamed as a tar straight into an S3 multipart upload. It contains:
- `images/`;
- `manifest.csv` (every effect value for each sheet);
- `augmentation_plan.csv`, `balance_by_condition.csv`, `effect_counts.json`;
- `record_manifest.csv`, `config.json`, `summary.json`;
- `failures.csv` (header only).

No local archive is written. The stream is hashed as it is sent and checked against S3's composite checksum; a mismatch deletes the upload. The `.sha256` sidecar is uploaded last and read back. The uploader is the clean12 render notebook's.

In [ ]:
import base64
import io
import threading
from collections import deque
from botocore.config import Config as BotoConfig

STREAM_PART_BYTES = 64 * 2**20            # 105 GiB -> ~1,700 parts (S3 max 10,000)
STREAM_BUFFER_BYTES = 4 * STREAM_PART_BYTES

class TarUploadStream(io.RawIOBase):
    """Pipe between tarfile (writer thread) and the S3 multipart upload (reader).

    Every byte is hashed once, in order, as it is written: the whole-archive
    SHA-256 for the sidecar, and per-part SHA-256s to check S3's composite
    checksum of the assembled object. read(n) blocks until n bytes or end of
    stream, so every S3 part except the last is exactly n bytes.
    """
    def __init__(self, part_bytes, max_buffered):
        super().__init__()
        assert max_buffered > part_bytes
        self.part_bytes, self.max_buffered = part_bytes, max_buffered
        self.digest = hashlib.sha256()
        self.part_digests = []
        self.bytes_written = 0
        self._part, self._part_fill = hashlib.sha256(), 0
        self._chunks, self._buffered = deque(), 0
        self._eof, self._error, self._aborted = False, None, False
        self._cv = threading.Condition()

    def readable(self):
        return True

    def writable(self):
        return True

    def tell(self):
        return self.bytes_written

    def _hash(self, data):
        self.digest.update(data)
        self.bytes_written += len(data)
        view = memoryview(data)
        while view:
            take = min(len(view), self.part_bytes - self._part_fill)
            self._part.update(view[:take])
            self._part_fill += take
            view = view[take:]
            if self._part_fill == self.part_bytes:
                self.part_digests.append(self._part.digest())
                self._part, self._part_fill = hashlib.sha256(), 0

    def write(self, data):
        data = bytes(data)
        with self._cv:
            while self._buffered >= self.max_buffered and not self._aborted:
                self._cv.wait()
            if self._aborted:
                raise IOError("S3 upload stopped; archive stream aborted")
        self._hash(data)  # single writer thread: hashing order == stream order
        with self._cv:
            self._chunks.append(data)
            self._buffered += len(data)
            self._cv.notify_all()
        return len(data)

    def read(self, size=-1):
        with self._cv:
            while not self._eof and (size is None or size < 0 or self._buffered < size):
                self._cv.wait()
            if self._error is not None:
                raise IOError(f"archive packing failed: {type(self._error).__name__}: {self._error}")
            want = self._buffered if size is None or size < 0 else min(size, self._buffered)
            out = bytearray()
            while len(out) < want:
                chunk = self._chunks.popleft()
                need = want - len(out)
                if len(chunk) > need:
                    self._chunks.appendleft(chunk[need:])
                    chunk = chunk[:need]
                out += chunk
            self._buffered -= len(out)
            self._cv.notify_all()
            return bytes(out)

    def finish(self, error=None):
        with self._cv:
            if self._part_fill:
                self.part_digests.append(self._part.digest())
                self._part_fill = 0
            self._eof, self._error = True, error
            self._cv.notify_all()

    def abort(self):
        with self._cv:
            self._aborted = True
            self._cv.notify_all()

    def s3_composite_sha256(self):
        """S3's COMPOSITE ChecksumSHA256 for these bytes uploaded as parts."""
        if len(self.part_digests) <= 1:
            return base64.b64encode(self.digest.digest()).decode()  # single PutObject
        combined = hashlib.sha256(b"".join(self.part_digests)).digest()
        return f"{base64.b64encode(combined).decode()}-{len(self.part_digests)}"

UPLOAD_VERIFIED = False
upload_prefix = urlparse(S3_UPLOAD_PREFIX)
if upload_prefix.scheme != "s3" or not upload_prefix.netloc:
    raise ValueError("S3_UPLOAD_PREFIX must be s3://bucket/ or s3://bucket/prefix/")
archive_name = f"{ARCHIVE_BASENAME}.{ARCHIVE_FORMAT}"
upload_bucket = upload_prefix.netloc
upload_key = "/".join(part for part in (upload_prefix.path.strip("/"), archive_name) if part)
uploaded_uri = f"s3://{upload_bucket}/{upload_key}"
image_prefix = OUTPUT_DIR.name + "/images/"
packed = Counter()

def _member_filter(info):
    name = info.name[len(OUTPUT_DIR.name) + 1:]
    if info.name.endswith(".part") or name.startswith("."):   # temp files, config lock
        packed["left_out"] += 1
        return None
    if info.isfile() and info.name.startswith(image_prefix) and info.name.endswith(".png"):
        packed["images"] += 1
    return info

def _pack(stream):
    try:
        with tarfile.open(fileobj=stream, mode="w", copybufsize=8 * 2**20) as archive:
            archive.add(OUTPUT_DIR, arcname=OUTPUT_DIR.name, recursive=True, filter=_member_filter)
        stream.finish()
    except BaseException as exc:
        stream.finish(exc)

upload_client = aws_session.client("s3", config=BotoConfig(
    retries={"max_attempts": 10, "mode": "standard"}, max_pool_connections=64))
stream_transfer = TransferConfig(multipart_threshold=STREAM_PART_BYTES, multipart_chunksize=STREAM_PART_BYTES,
                                 max_concurrency=32, use_threads=True, preferred_transfer_client="classic")
stream_transfer.max_in_memory_upload_chunks = 32  # ~2 GiB of parts in flight

stream = TarUploadStream(STREAM_PART_BYTES, STREAM_BUFFER_BYTES)
packer = threading.Thread(target=_pack, args=(stream,), name="tar-packer", daemon=True)
print(f"Streaming {OUTPUT_DIR.name} -> {uploaded_uri} ...", flush=True)
t0 = time.monotonic()
packer.start()
try:
    upload_client.upload_fileobj(stream, upload_bucket, upload_key,
                                 ExtraArgs={"ChecksumAlgorithm": "SHA256"}, Config=stream_transfer)
except BaseException:
    stream.abort()
    packer.join()
    raise
packer.join()
elapsed = time.monotonic() - t0
archive_sha256 = stream.digest.hexdigest()

remote = upload_client.head_object(Bucket=upload_bucket, Key=upload_key, ChecksumMode="ENABLED")
problems = []
if int(remote["ContentLength"]) != stream.bytes_written:
    problems.append(f"S3 size {int(remote['ContentLength']):,} != streamed {stream.bytes_written:,} bytes")
if remote.get("ChecksumSHA256") != stream.s3_composite_sha256():
    problems.append(f"S3 ChecksumSHA256 {remote.get('ChecksumSHA256')!r} != local {stream.s3_composite_sha256()!r}")
if packed["images"] != EXPECTED_RECORDS:
    problems.append(f"archive has {packed['images']:,} images; expected {EXPECTED_RECORDS:,}")
if problems:
    upload_client.delete_object(Bucket=upload_bucket, Key=upload_key)
    raise RuntimeError("Uploaded archive failed verification and was deleted:\n- " + "\n- ".join(problems))

sidecar_body = f"{archive_sha256}  {archive_name}\n".encode()
(WORK_DIR / (archive_name + ".sha256")).write_bytes(sidecar_body)
upload_client.put_object(Bucket=upload_bucket, Key=upload_key + ".sha256", Body=sidecar_body)
if upload_client.get_object(Bucket=upload_bucket, Key=upload_key + ".sha256")["Body"].read() != sidecar_body:
    raise RuntimeError("SHA-256 sidecar did not read back intact")
UPLOAD_VERIFIED = True
print(f"Uploaded {packed['images']:,} images; {stream.bytes_written / 2**30:.2f} GiB in "
      f"{len(stream.part_digests):,} parts, {elapsed / 60:.1f} min -> {uploaded_uri}")
print(f"S3 composite checksum verified: {remote['ChecksumSHA256']}")
print(f"SHA-256 {archive_sha256} -> {uploaded_uri}.sha256")
STAGES_DONE.add("upload")

## 11. Auto-terminate the instance

The instance terminates through the EC2 API only when **all** of these hold:
- `AUTO_TERMINATE` is on;
- §10 verified the upload;
- every pipeline cell (§2–§10, including the preflight and the benchmark) completed in this kernel since §1 was last run;
- **no cell raised an error** in that time. The error log in §1 records every failing cell, including cells re-run by hand after a fix. After any error, restart the kernel and Run All, or terminate the instance yourself.

Before terminating, the cell waits `TERMINATE_GRACE_SECONDS`; interrupting the kernel during that wait cancels. If the API call fails, the cell falls back to `sudo shutdown -h now`. That terminates the instance if its shutdown behaviour is *terminate*; otherwise it stops it.

Termination deletes the root EBS volume (the default *delete on termination*). The archive is already in S3 and verified at that point. Extra volumes you attached yourself may survive.

In [ ]:
REQUIRED_STAGES = ["install", "download", "kit", "records", "workers", "plan", "preflight",
                   "benchmark", "generate", "upload"]
missing_stages = [s for s in REQUIRED_STAGES if s not in STAGES_DONE]
blockers = []
if not AUTO_TERMINATE:
    blockers.append("AUTO_TERMINATE is False")
if not globals().get("UPLOAD_VERIFIED"):
    blockers.append("the upload was not verified in this kernel")
if missing_stages:
    blockers.append(f"these cells did not complete since §1: {missing_stages}")
if CELL_ERRORS:
    blockers.append(f"{len(CELL_ERRORS)} cell(s) raised an error since §1, e.g. {CELL_ERRORS[0]}")
if blockers:
    print("NOT terminating; the instance keeps running:\n- " + "\n- ".join(blockers))
else:
    identity = globals().get("instance") or ec2_identity()
    print(f"All {len(REQUIRED_STAGES)} pipeline cells completed without errors; upload verified: {uploaded_uri}")
    print(f"Terminating {identity['instance_id']} ({identity['instance_type']}, {identity['region']}) in "
          f"{TERMINATE_GRACE_SECONDS} s; interrupt the kernel to cancel.", flush=True)
    time.sleep(TERMINATE_GRACE_SECONDS)
    try:
        response = aws_session.client("ec2", region_name=identity["region"]).terminate_instances(
            InstanceIds=[identity["instance_id"]])
        print("Termination requested; instance state:",
              response["TerminatingInstances"][0]["CurrentState"]["Name"], flush=True)
    except Exception as exc:
        print(f"EC2 API termination failed ({type(exc).__name__}: {exc}); falling back to OS shutdown", flush=True)
        subprocess.run(["sudo", "shutdown", "-h", "now"], check=False)

## Notes

**Storage at peak:**

| Item | Size |
|---|---|
| Signal archive | 5.5 GiB |
| Extracted records | 13.5 GiB |
| Augmented PNGs (113,358 × ~1.45 MiB) | ~160 GiB |
| OS + packages | ~10 GiB |
| **Total** | **~190 GiB** |

**250 GB gp3** leaves headroom. Set the volume to 1,000 MB/s, so that §10 reads the 160 GiB in minutes rather than the ~25 min it takes at the 125 MB/s default.

**Memory.** Kit processes (~0.25 GiB) and augmentation (~0.6 GiB peak) never overlap within one worker, so 16 workers need about 10 GiB of the 32 GiB.

**matplotlib version.** The kit draws lead labels with matplotlib's default font, and the glyph pixels differ between matplotlib versions. With matplotlib 3.11 (installed by §2's unpinned `pip`), a re-render matched the clean12 sample sheets pixel for pixel; with 3.8.4, only the label text differed (about 4,600 pixels per sheet). Traces and grid are unaffected.

**Time on c8a.4xlarge (estimates; §8b measures).**

| Stage | Time |
|---|---|
| Download + extract | ~5 min |
| Render + augment | ~7–9 h |
| Upload | ~20–40 min |

**Using the output.**
- `manifest.csv` has one row per record: `record_path`, `image_path` and every effect value.
- Join to `record_manifest.csv` on `record_path` for labels and `patient_key`.
- There is no train/val/test split. Split by `patient_key`, using the same split as the clean sheets.
- Check a downloaded copy with `sha256sum -c clean_ecg_signal_images_clean12_augmented.tar.sha256`.

**Changing the design.** Edit `augment_worker.py` (`AUG_PALETTES`, `INK_BINS`, `TILT_BINS`, `PLAN_FACTORS`) and rebuild this notebook with `python3 make_notebook_account_a.py`. Use a new `OUTPUT_DIR` and `ARCHIVE_BASENAME`, because the config lock refuses to mix settings.